# MitoCore Curation 

The curation is carried out in 4 steps:

1. MitoCore original model: (parse and transfer annotations from the notes KEGG, BiGG, ec-codes)
2. Preliminary Curation: (Uniprot + units+ GPR )
3. MitoMammal: new reactions and respective genes and metabolites.
4. Alignment to Human1: (MitoMAMMAL + Human1 )

# Dependencies:

In [92]:
import cobra
import re
from urllib import request
from pprint import pprint
import pandas as pd
import memote
import copy
import gc
import pandas
import libsbml

import ssl #certificates needed to access KEGG API 
ssl._create_default_https_context = ssl._create_unverified_context #create the certificate

In [93]:
mitocore = cobra.io.read_sbml_model("/Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/MitoCore_Original_2017.xml")
doc = libsbml.readSBML('/Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/MitoCore_Original_2017.xml')
print(f"SBML Level: {doc.getLevel()}, Version: {doc.getVersion()}")

Model does not contain SBML fbc package information.
SBML package 'layout' not supported by cobrapy, information is not parsed
SBML package 'render' not supported by cobrapy, information is not parsed
Use of CHARGE in the notes element is discouraged, use fbc:charge instead: <Species M_10fthf_c "10-Formyltetrahydrofolate">
Use of FORMULA in the notes element is discouraged, use fbc:chemicalFormula instead: <Species M_10fthf_c "10-Formyltetrahydrofolate">
Use of CHARGE in the notes element is discouraged, use fbc:charge instead: <Species M_10fthf_m "10-Formyltetrahydrofolate">
Use of FORMULA in the notes element is discouraged, use fbc:chemicalFormula instead: <Species M_10fthf_m "10-Formyltetrahydrofolate">
Use of CHARGE in the notes element is discouraged, use fbc:charge instead: <Species M_13dpg_c "3-Phospho-D-glyceroyl phosphate">
Use of FORMULA in the notes element is discouraged, use fbc:chemicalFormula instead: <Species M_13dpg_c "3-Phospho-D-glyceroyl phosphate">
Use of CHARGE i

SBML Level: 2, Version: 1


functions to: 
1. Visualize structure of the original model
2. Cleaning-up false positive entries
During the curation process, placeholder entries such as N/A, nan, or None may have been introduced into the model. These values must be removed.
Note: MEMOTE testing tool interprets them as valid (non-empty) entries. As a result, MEMOTE classifies them as positive identifiers, which can give misleading results

In [94]:

def visualization_model_annotations(component):
    """"
    This function retrieves all elements inside 'annotations' as dictionary
    
    -model: COBRApy model object (e.g., mitocore)
    -component: str  ('reactions', 'metabolites', 'genes')
    """
    for comp in mitocore.__getattribute__(component):
        notes= comp.notes
        comp_complete= comp.__dict__
        print(notes)

# -----------------------------------------------------------------------------------------------------
def clean_invalid_annotations(model, invalid_values={'N/A', 'None', 'nan', 'NaN','n/a',''}):
    """
    Remove invalid placeholder values from model annotations across reactions, metabolites, and genes to avoid false positives in MEMOTE test.
    
    -model: COBRApy model object (already read with COBRApy, e.g. Mitocore)
    -invalid_values: dict, keys to be removed
    """
    #acces model's objects core-components 
    for component in model.reactions + model.metabolites + model.genes:
        #collect keys to remove
        keys_to_remove = []
        for key, val in component.annotation.items():
            if isinstance(val, (str, type(None))):
                #check if value inside key is in invalid dict
                if str(val).strip() in invalid_values:
                    keys_to_remove.append(key)
        for key in keys_to_remove:
            del component.annotation[key]
    return model

## Check reactions with multiple KEGG ids with structure: 
### R00267 (R00268 + R01899) 
### R08549 (R00621+R02570+R03316+R07618)

the reactions inside the parenthesis are equivalent to the reaction outside. Reaction divided in sub-steps 

In [95]:
#read SBML model
for comp in mitocore.__getattribute__('reactions'):
    notes= comp.notes
    kegg= notes.get('KEGG id', 'N/A')
    kegg_wrong= re.findall('^R\d+\s*\(\s*R\d+(?:\s*\+\s*R\d+)*\s*\)$', kegg)
    if kegg_wrong:
        print(f"Reaction {comp.id} has wrong KEGG id format: {kegg}")

Reaction PDHm has wrong KEGG id format: R00209 (R01699 + R02569 + R07618)
Reaction ACONTm has wrong KEGG id format: R01324 (R01325 + R01900)
Reaction ICDHyrm has wrong KEGG id format: R00267 (R00268 + R01899)
Reaction AKGDm has wrong KEGG id format: R08549 (R00621+R02570+R03316+R07618)
Reaction CI_MitoCore has wrong KEGG id format: R02163 (R00281 + R02166)
Reaction ICDHy has wrong KEGG id format: R00267 (R00268 + R01899)
Reaction ACONT has wrong KEGG id format: R01324 (R01325 + R01900)
Reaction ACSm has wrong KEGG id format: R00235 (R00316 + R00236)
Reaction ACS has wrong KEGG id format: R00235 (R00316 + R00236)


# MitoCore original model
(parse and transfer annotations from the notes KEGG, BiGG, ec-codes) |
(rewrite GPRs in readable form for AutoPACMEN) |



# Transfer infromation from reactions 'notes' in original Mitocore model to 'annotations'and standarize MIRIAM naming convention.

1. Transfer BiGG ids from 'notes' and add to 'annotations'
2. Transfer KEGG ids from 'notes' and add to 'annotations'
3. Transfer ec-codes from 'notes' and add to 'annotations'


In [96]:
# 1. extract ids from model  
def get_model_ids_for_databases(model, component_type, dict, databases, pattern):
    """
    Extract ids from model annotations for various databases across a specified component type.
    
    Parameters:
    - model: COBRApy model object (mitocore)
    - component_type: str, one of 'reactions', 'metabolites', 'genes'
    - dict: dictionary where database ids are located ('annotations', 'notes')
    - databases: list of str, e.g., ['kegg', 'bigg', 'metanetx', 'uniprot'].
    - pattern: regex pattern to extract ids from annotation strings. e.g for KEGG: r"R\d+"
    
    Returns:
    - DataFrame with component ID and extracted database ids.
    """
    # Initialize dictionary
    ids_dict = {'model_id': []}
    for db in databases: 
        ids_dict[db] = []

    # Access model core-component (reactions, genes, metabolites)
    components = getattr(model, component_type)

    for comp in components:
        ids_dict['model_id'].append(comp.id)

        for db in databases:
            ids= None
            value = None
            annotation = getattr(comp, dict, {})

            # look for different naming conventions 
            for key in [db, 
                        f"{db}.reaction", 
                        f"{db}.compound", 
                        f"{db}.metabolite", 
                        f"{db}.chemical", 
                        f"{db}.genes",
                        f"{db}.gene",
                        #reactions:'Recon2 id': 'r1447', 'KEGG id': 'R03857'
                        #metabolites: 'RECON2': '2oxoadp', 'KEGG ID': 'C00322'                        
                        f"{db} id",
                        f"{db.upper()} id",
                        f"{db.upper()} ID",
                        f"{db.upper()}"
                        ]:
                # search for ids and extract them as lis with re.findall 
                if key in annotation:
                    ids = re.findall(pattern, annotation[key])
                    if len(ids) == 0:
                        continue
                    break

            ids_dict[db].append(ids)

    return pd.DataFrame(ids_dict)
#-----------------------------------------------------------------------------------------------------------------------------
# 2. add them to current model
def add_ids_to_model(model, df, component_type, database):
    """
    Add ids from a DataFrame to the model annotations for a specified component type.

    Parameters:
    - model: COBRApy model object (e.g., mitocore)
    - df: DataFrame with model_id and database id columns.
    - component_type: str, one of 'reactions', 'metabolites', 'genes'
    - database: str, the name of the database column in df to map (e.g., 'kegg.reaction')
    
    Returns:
    - Updated model with new annotations added.
    """
    import ast
    import pandas as pd

    components = getattr(model, component_type)

    # iterate over dataframe rows
    for _, row in df.iterrows():
        model_id = row["model_id"]
        db_value = row[database]

        # normalize db_value 
        # skip missing values
        if db_value is None or (isinstance(db_value, float) and pd.isna(db_value)):
            continue

        # parse stringified lists (e.g. "['R00209', 'R01699']")
        if isinstance(db_value, str) and db_value.startswith("["):
            try:
                db_value = ast.literal_eval(db_value)
            except Exception as e:
                print(f"Error parsing {db_value}: {e}")
                continue

        # enforce list semantics
        if not isinstance(db_value, list):
            db_value = [db_value]

        if len(db_value) == 0:
            continue

        # update model 
        for comp in components:
            if comp.id == model_id:
                existing_annotation = comp.annotation.get(database)

                # normalize existing annotation to list
                if existing_annotation is None:
                    merged_values = db_value
                else:
                    if not isinstance(existing_annotation, list):
                        existing_annotation = [existing_annotation]
                    merged_values = list(
                        dict.fromkeys(existing_annotation + db_value)
                    )

                comp.annotation[database] = merged_values
                print(f"Added {merged_values} ({type(merged_values)}) to {model_id}")
                break

    return model



# Transfer annotations from reactions contained in Notes to Annotations with MIRIAM naming convention
1. BiGG
2. KEGG
3. ec-codes

In [97]:
#-----------------------------------------------------------------------------------------------------------------------------
#reactions:'Recon2 id': 'r1447', 'KEGG id': 'R03857'
# BiGG ids from notes for possible databases and collect in dataframe
print('geting ids for all possible databases ')
df_bigg_react = get_model_ids_for_databases(mitocore, 'reactions', 'notes', [ 'Recon2'], r"^[A-Za-z0-9_]+$")
print(df_bigg_react)

# Change column name in dataframe from 'Recon2' to 'bigg.reaction'. Required to match MIRIAM naming standards 
df_react_bigg = df_bigg_react.rename(columns={'model_id': 'model_id','KEGG':'kegg.reaction', 'Recon2': 'bigg.reaction'})
print(df_react_bigg)

# 2.Transfer BiGG ids from 'notes' and add to 'annotations'
print('adding bigg ids to the model')
bigg_react = add_ids_to_model(mitocore, df_react_bigg, 'reactions','bigg.reaction')

# ----------------------------------------------------------------------------------------------------------------------------
# KEGG ids from notes for possible databases and collect in dataframe
print('geting ids for all possible databases ')
df_kegg_react = get_model_ids_for_databases(mitocore, 'reactions', 'notes', ['KEGG'],r"R\d+")
print(df_kegg_react)

# Change column name in dataframe from 'KEGG id' to 'kegg.reaction'. Required to match MIRIAM naming standards 
df_react_kegg = df_kegg_react.rename(columns={'model_id': 'model_id','KEGG':'kegg.reaction', 'Recon2': 'bigg.reaction'})
print(df_react_kegg)

# 2.Transfer KEGG ids from 'notes' and add to 'annotations'
print('adding kegg ids to the model')
kegg_react = add_ids_to_model(mitocore, df_react_kegg, 'reactions','kegg.reaction')

# ----------------------------------------------------------------------------------------------------------------------------
#  ec-codes from notes for possible databases and collect in dataframe
print('geting ec-codes from notes ')
df_ec_react = get_model_ids_for_databases(mitocore, 'reactions', 'notes', ['EC Number'], r"\d+\.\d+\.\d+\.\d+")
print(df_ec_react)  
# Change column name in dataframe from 'EC Number' to 'ec-code'. Required to match MIRIAM naming standards
df_react_ec = df_ec_react.rename(columns={'model_id': 'model_id','EC Number':'ec-code'})
print(df_react_ec) 
# 2.Transfer ec-codes from 'notes' and add to 'annotations'
print('adding ec-codes to the model')
ec_react = add_ids_to_model(mitocore, df_react_ec, 'reactions','ec-code')   


geting ids for all possible databases 
       model_id        Recon2
0      EX_2hb_e          None
1       EX_ac_e          None
2     EX_acac_e          None
3      EX_akg_e          None
4    EX_ala_B_e          None
..          ...           ...
550         COt         [COt]
551         NOt         [NOt]
552  PCHOLHSTDe  [PCHOLHSTDe]
553        PSt3        [PSt3]
554         PEt         [PEt]

[555 rows x 2 columns]
       model_id bigg.reaction
0      EX_2hb_e          None
1       EX_ac_e          None
2     EX_acac_e          None
3      EX_akg_e          None
4    EX_ala_B_e          None
..          ...           ...
550         COt         [COt]
551         NOt         [NOt]
552  PCHOLHSTDe  [PCHOLHSTDe]
553        PSt3        [PSt3]
554         PEt         [PEt]

[555 rows x 2 columns]
adding bigg ids to the model
Added ['HEX1'] (<class 'list'>) to HEX1
Added ['G6PPer'] (<class 'list'>) to G6PPer
Added ['PGI'] (<class 'list'>) to PGI
Added ['PFK'] (<class 'list'>) to PFK
Adde

# Rewritte GPRs and delete malformed GPR (N/A, unknown)

## Drop malformed genes

In [98]:
# occurred during gpr parsing, GENE_LIST did not have the same length as HGNC ids, UCP3 was missing
mitocore.reactions.HtmB_MitoCore.notes['GENE_LIST'] = "UCP2 or UCP3"

# drop genes not representing anything
cobra.manipulation.delete.remove_genes(
    mitocore, 
    ["Non-enzymatic", "Non-Enzymatic", "Unknown", "N/A"], remove_reactions = False)

###  Rewrite complex GPR rules
problematic GPRs (discovered during "get_initial_spreadsheet" and debugging of the same):

GPR occurs in reactions: CV_MitoCore

ENSG00000152234 and ENSG00000110955 and ENSG00000165629 and ENSG00000099624 and ENSG00000124172 and ENSG00000116459 and (ENSG00000159199 or ENSG00000135390 or ENSG00000154518) and ENSG00000167863 and ENSG00000169020 and ENSG00000154723 and ENSG00000241468 and ENSG00000167283 and ENSG00000249222 and ENSG00000241837 and ENSG00000198899 and ENSG00000228253

GPR occurs in reactions: PCFLOPm, PSFLIPm, PEFLIPm

(ENSG00000124406 or ENSG00000143515 or ENSG00000081923 or ENSG00000104043 or ENSG00000054793 or ENSG00000166377 or ENSG00000206190 or ENSG00000145246 or ENSG00000068650 or ENSG00000058063 or ENSG00000101974) and (ENSG00000112697 or ENSG00000182107)

These rules cannot be split correctly by autoPACMEN and would be ignored. Therefore, they are rewritten.

In [99]:
#variable subunits are added at the end of every complex
CV_MitoCore_gpr = "(ENSG00000152234 and ENSG00000110955 and ENSG00000165629 and ENSG00000099624 and ENSG00000124172 and ENSG00000116459 and ENSG00000167863 and ENSG00000169020 and ENSG00000154723 and ENSG00000241468 and ENSG00000167283 and ENSG00000249222 and ENSG00000241837 and ENSG00000198899 and ENSG00000228253 and ENSG00000159199) or (ENSG00000152234 and ENSG00000110955 and ENSG00000165629 and ENSG00000099624 and ENSG00000124172 and ENSG00000116459 and ENSG00000167863 and ENSG00000169020 and ENSG00000154723 and ENSG00000241468 and ENSG00000167283 and ENSG00000249222 and ENSG00000241837 and ENSG00000198899 and ENSG00000228253 and ENSG00000135390) or (ENSG00000152234 and ENSG00000110955 and ENSG00000165629 and ENSG00000099624 and ENSG00000124172 and ENSG00000116459 and ENSG00000167863 and ENSG00000169020 and ENSG00000154723 and ENSG00000241468 and ENSG00000167283 and ENSG00000249222 and ENSG00000241837 and ENSG00000198899 and ENSG00000228253 and ENSG00000154518)"

# iterate through all combinations of subunits
first_subunits = ["ENSG00000124406", "ENSG00000143515", "ENSG00000081923", "ENSG00000104043", "ENSG00000054793", "ENSG00000166377", "ENSG00000206190", "ENSG00000145246", "ENSG00000068650", "ENSG00000058063", "ENSG00000101974"]
second_subunits = ["ENSG00000112697", "ENSG00000182107"]

PCFLOPm_PSFLIPm_PEFLIPm_gpr = ""

 #Discuss with Emanuel the output! shouldn't it work with a While conditional??
count = 1

for first_subunit in first_subunits:
    for second_subunit in second_subunits:
        PCFLOPm_PSFLIPm_PEFLIPm_gpr += "(" + first_subunit + " and " + second_subunit + ")"
        #If not the last combination, " or " is appended to separate subunit pairs.
        if count < len(first_subunits) * len(second_subunits):
            PCFLOPm_PSFLIPm_PEFLIPm_gpr += " or "
        count += 1
        print(PCFLOPm_PSFLIPm_PEFLIPm_gpr)

(ENSG00000124406 and ENSG00000112697) or 
(ENSG00000124406 and ENSG00000112697) or (ENSG00000124406 and ENSG00000182107) or 
(ENSG00000124406 and ENSG00000112697) or (ENSG00000124406 and ENSG00000182107) or (ENSG00000143515 and ENSG00000112697) or 
(ENSG00000124406 and ENSG00000112697) or (ENSG00000124406 and ENSG00000182107) or (ENSG00000143515 and ENSG00000112697) or (ENSG00000143515 and ENSG00000182107) or 
(ENSG00000124406 and ENSG00000112697) or (ENSG00000124406 and ENSG00000182107) or (ENSG00000143515 and ENSG00000112697) or (ENSG00000143515 and ENSG00000182107) or (ENSG00000081923 and ENSG00000112697) or 
(ENSG00000124406 and ENSG00000112697) or (ENSG00000124406 and ENSG00000182107) or (ENSG00000143515 and ENSG00000112697) or (ENSG00000143515 and ENSG00000182107) or (ENSG00000081923 and ENSG00000112697) or (ENSG00000081923 and ENSG00000182107) or 
(ENSG00000124406 and ENSG00000112697) or (ENSG00000124406 and ENSG00000182107) or (ENSG00000143515 and ENSG00000112697) or (ENSG00000

In [100]:
mitocore.reactions.CV_MitoCore.gene_reaction_rule = CV_MitoCore_gpr
mitocore.reactions.PCFLOPm.gene_reaction_rule = PCFLOPm_PSFLIPm_PEFLIPm_gpr
mitocore.reactions.PSFLIPm.gene_reaction_rule = PCFLOPm_PSFLIPm_PEFLIPm_gpr
mitocore.reactions.PEFLIPm.gene_reaction_rule = PCFLOPm_PSFLIPm_PEFLIPm_gpr

# Transfer annotations from metabolites contained in Notes to Annotations with MIRIAM naming convention

2.  BiGG 
4.  KEGG

In [101]:
#-----------------------------------------------------------------------------------------------------------------------------
#reactions:'Recon2 id': 'r1447', 'KEGG id': 'R03857'
# BiGG ids from notes and collect in dataframe
print('geting ids for all possible databases ')
df_bigg_met = get_model_ids_for_databases(mitocore, 'metabolites', 'notes', [ 'Recon2'], r"^[A-Za-z0-9_]+$")
print(df_bigg_met)

# Change column name in dataframe from 'Recon2' to 'bigg.metabolite'. Required to match MIRIAM naming standards 
df_met_bigg = df_bigg_met.rename(columns={'model_id': 'model_id', 'Recon2': 'bigg.metabolite'})
print(df_met_bigg)

# Transfer BiGG ids from 'notes' and add to 'annotations'
print('adding bigg ids to the model')
bigg_met = add_ids_to_model(mitocore, df_met_bigg, 'metabolites','bigg.metabolite')

# ----------------------------------------------------------------------------------------------------------------------------

#reactions:'Recon2 id': 'r1447', 'KEGG id': 'R03857'
# KEGG ids from notes and collect in dataframe
print('geting ids for all possible databases ')
df_kegg_met = get_model_ids_for_databases(mitocore, 'metabolites', 'notes', [ 'KEGG ID'], r"C\d+")
print(df_kegg_met)

# Change column name in dataframe from 'KEGG ID' to 'kegg.compound'. Required to match MIRIAM naming standards 
df_met_kegg = df_kegg_met.rename(columns={'model_id': 'model_id', 'KEGG ID': 'kegg.compound'})
print(df_met_kegg)

# Transfer BiGG ids from 'notes' and add to 'annotations'
print('adding bigg ids to the model')
kegg_met = add_ids_to_model(mitocore, df_met_kegg, 'metabolites','kegg.compound')

geting ids for all possible databases 
       model_id      Recon2
0      10fthf_c    [10fthf]
1      10fthf_m    [10fthf]
2       13dpg_c     [13dpg]
3    1pipdn2c_c  [1pipdn2c]
4      1pyr5c_m    [1pyr5c]
..          ...         ...
436   myrsACP_c   [myrsACP]
437   HC01605_c   [HC01605]
438   HC01326_c   [HC01326]
439   HC01606_c   [HC01606]
440   palmACP_c   [palmACP]

[441 rows x 2 columns]
       model_id bigg.metabolite
0      10fthf_c        [10fthf]
1      10fthf_m        [10fthf]
2       13dpg_c         [13dpg]
3    1pipdn2c_c      [1pipdn2c]
4      1pyr5c_m        [1pyr5c]
..          ...             ...
436   myrsACP_c       [myrsACP]
437   HC01605_c       [HC01605]
438   HC01326_c       [HC01326]
439   HC01606_c       [HC01606]
440   palmACP_c       [palmACP]

[441 rows x 2 columns]
adding bigg ids to the model
Added ['10fthf'] (<class 'list'>) to 10fthf_c
Added ['10fthf'] (<class 'list'>) to 10fthf_m
Added ['13dpg'] (<class 'list'>) to 13dpg_c
Added ['1pipdn2c'] (<class '


# Transfer annotations from genes contained in Notes to Annotations with MIRIAM naming convention

1.  HGNC

In [102]:
# statistics on gene annotations and mapping of gene code in gpr to HGNC codes and gene names

# 1. splits word separated by "and", "or", "/", " " into strings in a list
def split_gene_rule(gene_string):
    """Strips all brackets and "and" and "or" from gene rule and splits it into a list of gene codes"""
    space_split = gene_string.split(" ")

    if len(space_split) == 1:
        return space_split

    space_split = [re.sub(r'[\(\),]', '', string) for string in space_split]

    split = [string for string in space_split if not string == "or" and not string == "and"]

    return split
#-----------------------------------------------------------------------------------------------------------------------------
# parse reaction notes to get gene annotations 
def parse_reaction(reaction, gene_hgnc_mapping, reaction_stats):

        notes = reaction.notes
    
        ensebml_annotation_string = "" #to store GENE_ASSOCIATION ids
        gene_hgnc_string = "" #to store reaction id with hgnc gene code
        gene_name_string = "" # to store GENE_LIST

        non_gene_annotations = ["Unknown", "N/A", "Non-Enzymatic", "Non-enzymatic"]

        #parse notes 
        #all stored in "reaction_stats" is for statistical counting 
        for key in notes.keys():
            reaction_stats["annotations"].add(key)
            # count reactions with hgnc gene code
            if key == "HGNC" or key == "(HGNC":
                reaction_stats["reactions_with_hgnc"].add(reaction.id)
                gene_hgnc_string = notes[key]                

            # count reactions with gene associations
            # handle case of non_gene_annotations where "GENE_ASSOCIATION" key exists but is N/A or unknown
            if key == "GENE_ASSOCIATION" and not (notes[key] in non_gene_annotations):
                reaction_stats["reactions_with_genes"].add(reaction.id)

                ensebml_annotation_string = notes[key]

            if key == "GENE_LIST":
                gene_name_string = notes[key]
        
        # check how many reactions have gene and hgnc code (pure statistic quantification)
        if gene_hgnc_string == "":
            reaction_stats["reaction_without_hgnc"].add(reaction.id)
        if gene_hgnc_string == "" and ensebml_annotation_string == "":
             reaction_stats["reaction_without_genes"].add(reaction.id)

        ensembl_code = []
        hgnc_ids = []
        hgnc_symbols = []

        #if not emoty string in following cases, process the IDs with split_gene_rule() function
        if not ensebml_annotation_string == "":
            ensembl_code = split_gene_rule(ensebml_annotation_string)
            
        if not gene_hgnc_string == "":
            hgnc_ids = split_gene_rule("HGNC:" + gene_hgnc_string)

        if not gene_name_string == "":
            hgnc_symbols = split_gene_rule(gene_name_string)

        #dictionary to store parameters obtained from hgnc_ids
        for ensembl, hgnc_id, hgnc_symbol in zip(ensembl_code, hgnc_ids, hgnc_symbols):
            gene_hgnc_mapping[ensembl] = {
                "hgnc": hgnc_id,
                "hgnc.symbol": hgnc_symbol,
                "ensembl": ensembl
            }
#-----------------------------------------------------------------------------------------------------------------------------
# 3. Main function to get gene annotations from reactions
def get_gene_annotations_from_reactions(reactions):

    '''Parses notes in reaction objects, retrieves uniprot ids for each gene id in the gpr rules and returns dict containing mapping of gene id to gene name, hgnc id and uniprot id'''
    # gene code in gpr -> hgnc, gene name
    gene_hgnc_mapping = {}

    reaction_stats = {
        "annotations": set(),           # all annotations in reactions
        "reactions_with_genes": set(),  # reactions with gene rules
        "reactions_with_hgnc": set(),   # reactions with hgnc gene codes
        "reaction_without_hgnc": set(), # reactions without hgnc gene codes
        "reaction_without_genes": set() # reactions without gene rules
    }
    #
    reaction_index = 0
    for reaction in reactions:
        print("                                                                       ", end="\r")
        print("Parsing reaction " + str(reaction_index) + "/" + str(len(reactions)) + "(reaction id: " + reaction.id + ")", sep=" ", end='\r')
        parse_reaction(reaction, gene_hgnc_mapping, reaction_stats)
        reaction_index += 1

    print("occuring annotations: " + str(reaction_stats["annotations"]))

    print("reactions with gene rule / reactions with hgnc ids: " + str(len(reaction_stats["reactions_with_genes"])) + "/" + str(len(reaction_stats["reactions_with_hgnc"])))

    return gene_hgnc_mapping



In [103]:
# add gene annotations 
# MIRIAM annotaions were only included in the notes of the reactions
# Mitocore -> MIRIAM
# HGNC -> hgnc (id)
# GENE_LIST -> hgnc.symbol
# Ensembl -> ensembl

# Added annotations

gene_id_annotation_dict = get_gene_annotations_from_reactions(mitocore.reactions)

for gene in mitocore.genes:
    if gene.id in gene_id_annotation_dict.keys():
        gene._annotation = gene_id_annotation_dict[gene.id]

occuring annotations: {'Recon2 id', 'SUBSYSTEM', 'HGNC', 'MitoCarta', 'HPA mRNA level in heart tissue', 'HPA protein level in heart tissue', 'Gene name', 'Directionality rationale', 'MitoCarta score', 'EC Number', 'GENE_LIST', 'Recon2 formula', 'KEGG id', 'Description', 'Model Reaction Number', 'GENE_ASSOCIATION', '(HGNC'}
reactions with gene rule / reactions with hgnc ids: 353/353


In [104]:
# load and clean model from N/A values
model_clean = clean_invalid_annotations(mitocore)
print(type(model_clean))
# save cleaned model as new SBML file
cobra.io.write_sbml_model(model_clean, "/Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/MitoCore_Curation/Mitocore_Original.xml")
# Memote report after cleaning model to check if positive Memote features remain the same or false positive were present
!memote report snapshot --filename "/Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/MitoCore_Curation/Mitocore_Original.html" /Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/MitoCore_Curation/Mitocore_Original.xml

<class 'cobra.core.model.Model'>
The current solver interface glpk doesn't support setting the optimality tolerance.
============================= test session starts ==============================
platform darwin -- Python 3.10.8, pytest-7.1.2, pluggy-1.0.0
rootdir: /Users/benjaminreyes
plugins: anyio-3.5.0, typeguard-4.4.2
collected 155 items / 1 skipped                                                

../../../../../../anaconda3/lib/python3.10/site-packages/memote/suite/tests/test_annotation.py F [  0%]
F.FFFFFFFFFFFFFFFFFFFFFFFFFFFFFFF.FFFFFFF.FF.FF.F.FFFFFFFFFFFF..         [ 41%]
../../../../../../anaconda3/lib/python3.10/site-packages/memote/suite/tests/test_basic.py . [ 42%]
.....FF.......F.F.F.FF                                                   [ 56%]
../../../../../../anaconda3/lib/python3.10/site-packages/memote/suite/tests/test_biomass.py . [ 57%]
FFFFFF....FFFFF.FF                                                       [ 69%]
../../../../../../anaconda3/lib/python3.10/site-

In [105]:
for reactions in mitocore.reactions:
    annotations = reactions.annotation
    # print(annotations)
    for keys, vals in annotations.items():
        if keys == 'kegg.reaction' and isinstance(vals, list) and 'R03174' in vals:
            print(annotations)

{'bigg.reaction': ['OIVD3m'], 'kegg.reaction': ['R07603', 'R07604', 'R03174', 'R07618'], 'ec-code': ['1.2.4.4', '2.3.1.168', '1.8.1.4']}


# ---------------------------------------------------------------

# MitoCore preliminary curation to meet AutoPACMEN minimal requirements

Adjust constraint units and add Uniprot identifiers for genes


# 0) Adjust constraint units
- Mitocore fluxes are calculated in µmol/min/gDW, mmol/h/gDW is the standard unit
- all constraints need to be adjusted, i.e., multiplied with 0.06

$$
1 \frac{µmol}{min*g_{DW}} = \frac{1}{1000} \frac{mmol}{µmol} * µmol * \frac{1}{min} * 60 \frac{min}{h} * \frac{1}{g_{DW}} = \frac{60}{1000} \frac{mmol}{h*g_{DW}} = 0.06 \frac{mmol}{h*g_{DW}}
$$

- constraints are rewritten in set_parameters.ipynb

In [106]:
for reaction in mitocore.reactions:
    ub = reaction.upper_bound
    lb = reaction.lower_bound
    if not ub == 1000: # keep default constraints
        reaction.upper_bound = ub * 0.06 # factor 0.6 for unit convertion umol/min gDW -> mmol/h gDW  
    if not lb == -1000:
        reaction.lower_bound = lb * 0.06
    #why keeping default bounds and changing them just if not == 1000?

    # check if any ubs are -1000 or lbs are 1000 (just to be save)
    if reaction.upper_bound == -1000 or reaction.lower_bound == 1000:
        print(reaction.id)

Manual curation of reactions with mutiple bigg ids

In [107]:
for met in mitocore.metabolites:
    if met.id == "2hb_c":
        print(f"{met.id}",met.annotation)
    if met.id == "2hb_e":
        print(f"{met.id}",met.annotation)
    if met.id == "3hbcoa_m":
        print(f"{met.id}",met.annotation)
    if met.id == "3hibutcoa_m":
        print(f"{met.id}",met.annotation) 
    if met.id == "dd2coa_m":
        print(f"{met.id}",met.annotation)

2hb_c {'bigg.metabolite': ['2hb'], 'kegg.compound': ['C05984']}
2hb_e {'bigg.metabolite': ['2hb'], 'kegg.compound': ['C05984']}
3hbcoa_m {'bigg.metabolite': ['3hbcoa'], 'kegg.compound': ['C01144']}
3hibutcoa_m {'bigg.metabolite': ['3hibutcoa'], 'kegg.compound': ['C06000']}
dd2coa_m {'bigg.metabolite': ['dd2coa'], 'kegg.compound': ['C03221']}


# Add Uniprot to Gene annotations 

HGNC --> Uniprot (API)

In [108]:
# 2. Takes an HGNC ID to retrieve its corresponding UniProt ID
%pip install requests
import requests

def get_uniprot_from_hgnc(df):

    ids={'model_id':[], 'hgnc':[], 'uniprot':[]}
    
    model_id = df.loc[df['hgnc'] == hgnc_id, 'model_id'].values[0]
    #convert list to string for API request
    df['hgnc'] = df['hgnc'].str[0]

    for hgnc_id in df['hgnc']:
        uniprot_acc = None

        hgnc_fetch_id = "https://rest.genenames.org/fetch/hgnc_id/" + hgnc_id #construct request URL + HGNC ID 

        resp = requests.get(hgnc_fetch_id, headers={"Accept": "application/json"}) #send URL request 

        if resp.ok:
            docs = resp.json()["response"]["docs"]

            if docs:
                json_doc = docs[0]
                try:
                    uniprot_acc = json_doc["uniprot_ids"][0]
                    if len(json_doc["uniprot_ids"]) > 1:
                        print(hgnc_id + " has multiple uniprot ids")

                except KeyError:
                    print(hgnc_id + " has no uniprot id")

        ids['hgnc'].append(hgnc_id)
        ids['uniprot'].append(uniprot_acc) 
        ids['model_id'].append(model_id)
    return pd.DataFrame(ids)


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [109]:
def get_uniprot_from_hgnc(df):

    ids = {'model_id': [], 'hgnc': [], 'uniprot': []}

    for index, row in df.iterrows():

        model_id = row['model_id']
        hgnc_id = row['hgnc'][0]   # extract from list
        uniprot_acc = None

        url = f"https://rest.genenames.org/fetch/hgnc_id/{hgnc_id}"
        resp = requests.get(url, headers={"Accept": "application/json"})

        if resp.ok:
            docs = resp.json()["response"]["docs"]
            if docs and "uniprot_ids" in docs[0]:
                uniprot_acc = docs[0]["uniprot_ids"][0]
                if len(docs[0]["uniprot_ids"]) > 1:
                    print(f"{hgnc_id} has multiple UniProt IDs")

        ids['model_id'].append(model_id)
        ids['hgnc'].append(hgnc_id)
        ids['uniprot'].append(uniprot_acc)

    return pd.DataFrame(ids)

In [110]:
df_hgnc_genes = get_model_ids_for_databases(mitocore, 'genes', 'annotation', [ 'hgnc'], r"HGNC:\d+")
print(df_hgnc_genes)


            model_id          hgnc
0    ENSG00000159399   [HGNC:4923]
1    ENSG00000106633   [HGNC:4195]
2    ENSG00000156515   [HGNC:4922]
3    ENSG00000160883   [HGNC:4925]
4    ENSG00000131482   [HGNC:4056]
..               ...           ...
382  ENSG00000124406  [HGNC:13531]
383  ENSG00000143515  [HGNC:13534]
384  ENSG00000101974  [HGNC:13554]
385  ENSG00000166377  [HGNC:13541]
386  ENSG00000081923   [HGNC:3706]

[387 rows x 2 columns]


In [111]:
df_uniprot_genes= get_uniprot_from_hgnc(df_hgnc_genes)
print(df_uniprot_genes)

HGNC:7459 has multiple UniProt IDs
            model_id        hgnc uniprot
0    ENSG00000159399   HGNC:4923  P52789
1    ENSG00000106633   HGNC:4195  P35557
2    ENSG00000156515   HGNC:4922  P19367
3    ENSG00000160883   HGNC:4925  P52790
4    ENSG00000131482   HGNC:4056  P35575
..               ...         ...     ...
382  ENSG00000124406  HGNC:13531  Q9Y2Q0
383  ENSG00000143515  HGNC:13534  P98198
384  ENSG00000101974  HGNC:13554  Q8NB49
385  ENSG00000166377  HGNC:13541  O43861
386  ENSG00000081923   HGNC:3706  O43520

[387 rows x 3 columns]


In [112]:
kegg_met = add_ids_to_model(mitocore, df_uniprot_genes, 'genes','uniprot')

Added ['P52789'] (<class 'list'>) to ENSG00000159399
Added ['P35557'] (<class 'list'>) to ENSG00000106633
Added ['P19367'] (<class 'list'>) to ENSG00000156515
Added ['P52790'] (<class 'list'>) to ENSG00000160883
Added ['P35575'] (<class 'list'>) to ENSG00000131482
Added ['P06744'] (<class 'list'>) to ENSG00000105220
Added ['P08237'] (<class 'list'>) to ENSG00000152556
Added ['P09467'] (<class 'list'>) to ENSG00000165140
Added ['O00757'] (<class 'list'>) to ENSG00000130957
Added ['P09972'] (<class 'list'>) to ENSG00000109107
Added ['P04075'] (<class 'list'>) to ENSG00000149925
Added ['P60174'] (<class 'list'>) to ENSG00000111669
Added ['P04406'] (<class 'list'>) to ENSG00000111640
Added ['P00558'] (<class 'list'>) to ENSG00000102144
Added ['P15259'] (<class 'list'>) to ENSG00000164708
Added ['P18669'] (<class 'list'>) to ENSG00000171314
Added ['P13929'] (<class 'list'>) to ENSG00000108515
Added ['P09104'] (<class 'list'>) to ENSG00000111674
Added ['P06733'] (<class 'list'>) to ENSG00000

In [113]:
counter=0
for gene in mitocore.genes:
    if 'uniprot' in gene.annotation:
        print(f"{gene.id}: {gene.annotation['uniprot']}")
        counter += 1
print(f"Total genes with uniprot ids: {counter}")

ENSG00000159399: ['P52789']
ENSG00000106633: ['P35557']
ENSG00000156515: ['P19367']
ENSG00000160883: ['P52790']
ENSG00000131482: ['P35575']
ENSG00000105220: ['P06744']
ENSG00000152556: ['P08237']
ENSG00000165140: ['P09467']
ENSG00000130957: ['O00757']
ENSG00000109107: ['P09972']
ENSG00000149925: ['P04075']
ENSG00000111669: ['P60174']
ENSG00000111640: ['P04406']
ENSG00000102144: ['P00558']
ENSG00000164708: ['P15259']
ENSG00000171314: ['P18669']
ENSG00000108515: ['P13929']
ENSG00000111674: ['P09104']
ENSG00000074800: ['P06733']
ENSG00000067225: ['P14618']
ENSG00000124253: ['P35558']
ENSG00000111716: ['P07195']
ENSG00000160211: ['P11413']
ENSG00000130313: ['O95336']
ENSG00000142657: ['P52209']
ENSG00000153574: ['P49247']
ENSG00000235376: ['Q2QD12']
ENSG00000163931: ['P29401']
ENSG00000177156: ['P37837']
ENSG00000091140: ['P09622']
ENSG00000150768: ['P10515']
ENSG00000131828: ['P08559']
ENSG00000168291: ['P11177']
ENSG00000062485: ['O75390']
ENSG00000100412: ['Q99798']
ENSG00000166411: ['P

In [114]:
# load and clean model from N/A values
model_clean = clean_invalid_annotations(mitocore)
print(type(model_clean))
# save cleaned model as new SBML file
cobra.io.write_sbml_model(model_clean, "/Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/MitoCore_Curation/Mitocore_Preliminary.xml")
# Memote report after cleaning model to check if positive Memote features remain the same or false positive were present
!memote report snapshot --filename "/Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/MitoCore_Curation/Mitocore_Preliminary.html" /Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/MitoCore_Curation/Mitocore_Preliminary.xml

<class 'cobra.core.model.Model'>
The current solver interface glpk doesn't support setting the optimality tolerance.
============================= test session starts ==============================
platform darwin -- Python 3.10.8, pytest-7.1.2, pluggy-1.0.0
rootdir: /Users/benjaminreyes
plugins: anyio-3.5.0, typeguard-4.4.2
collected 155 items / 1 skipped                                                

../../../../../../anaconda3/lib/python3.10/site-packages/memote/suite/tests/test_annotation.py F [  0%]
F.FFFFFFFFFFFFFFFFFFFFF.FFFFFFFFF.FFFFFFF.FF.FF.F.FFF.FFFFFFFF..         [ 41%]
../../../../../../anaconda3/lib/python3.10/site-packages/memote/suite/tests/test_basic.py . [ 42%]
.....FF.......F.F.F.FF                                                   [ 56%]
../../../../../../anaconda3/lib/python3.10/site-packages/memote/suite/tests/test_biomass.py . [ 57%]
FFFFFF....FFFFF.FF                                                       [ 69%]
../../../../../../anaconda3/lib/python3.10/site-

# Add MitoMAMMAL GPRs and reactions to MitoCore

In [115]:
mitocore= cobra.io.read_sbml_model("/Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/MitoCore_Curation/Mitocore_Preliminary.xml")
mitomammal= cobra.io.read_sbml_model("/Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/MitoMAMMAL_08.25.xml")

In [116]:
mitomammal_in_mitocore= {'mitocore_id':[], 'mitomammal_id':[], 'mitocore_GPR':[], 'mitomammal_GPR':[]}
mitomammal_not_in_mitocore= { 'mitomammal_id':[], 'mitomammal_GPR':[]}


for reaction in mitomammal.reactions:
    if reaction.id in mitocore.reactions:
            #mitomammal
            mitomammal_reaction = mitocore.reactions.get_by_id(reaction.id)
            mitmammal_GPR = reaction.gene_reaction_rule
            #mitocore
            mitocore_reaction = mitocore.reactions.get_by_id(reaction.id)
            mitocore_GPR = mitocore_reaction.gene_reaction_rule

            mitomammal_in_mitocore['mitocore_id'].append(mitocore_reaction.id)
            mitomammal_in_mitocore['mitomammal_id'].append(reaction.id)
            mitomammal_in_mitocore['mitocore_GPR'].append(mitocore_GPR)
            mitomammal_in_mitocore['mitomammal_GPR'].append(mitmammal_GPR)

    elif reaction.id not in mitocore.reactions:
            mitomammal_not_in_mitocore['mitomammal_id'].append(reaction.id)
            mitomammal_not_in_mitocore['mitomammal_GPR'].append(reaction.gene_reaction_rule)
            


mitomammal_in_mitocore_df= pd.DataFrame(mitomammal_in_mitocore)
mitomammal_not_in_mitocore_df= pd.DataFrame(mitomammal_not_in_mitocore)




mitocore_not_in_mitomammal= { 'mitomammal_id':[], 'mitomammal_GPR':[]}
for reaction in mitocore.reactions:
       if reaction not in mitomammal.reactions:
            mitocore_reaction = mitocore.reactions.get_by_id(reaction.id)
            mitocore_GPR = mitocore_reaction.gene_reaction_rule
            mitocore_not_in_mitomammal['mitomammal_id'].append(reaction.id)
            mitocore_not_in_mitomammal['mitomammal_GPR'].append(mitocore_GPR)
print(f"Number of reactions in mitocore but not in mitomammal: {len(mitocore_not_in_mitomammal)}")
mitocore_not_in_mitomammal_df= pd.DataFrame(mitocore_not_in_mitomammal)


Number of reactions in mitocore but not in mitomammal: 2


# check manually new MitoMAMMAl reactions:
1. CBPS
2. ASPCT
3. DHORTS
4. DHORD9
5. DM_orot_c

In [117]:

reactions=['CBPS', 'ASPCT', 'DHORTS', 'DHORD9', 'DM_orot_c'] #list of  5 new reactions in MitoMAMMAL
genes_mitomammal= set([])
metabolites_mitomammal= set([])

#
for reaction in mitomammal.reactions:
    if reaction.id in reactions:
        annotations = reaction.annotation
        notes= reaction.notes
        metabolites= reaction.metabolites
        genes= reaction.genes
        
        print("--------------------------------------------------")
        print(reaction.id)
        print("--------------------------------------------------")
        for metabolite in metabolites:
            metabolites_mitomammal.add(metabolite)
            #print(f"for reaction{reaction.id} metabolite.id: {metabolite.id} annotation {metabolite.annotation}, formula {metabolite.formula}")
        print("--------------------------------------------------")
        for gene in genes:
            genes_mitomammal.add(gene)
            #print(f"for reaction{reaction.id} gene.id: {gene.id}", gene.annotation)



--------------------------------------------------
CBPS
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
ASPCT
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
DHORTS
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
DHORD9
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
DM_orot_c
--------------------------------------------------
--------------------------------------------------


In [118]:
print(genes_mitomammal)
print(metabolites_mitomammal)


{<Gene ENSMUSG00000013629 at 0x29cc34590>, <Gene ENSMUSG00000031730 at 0x29cc49810>, <Gene ENSG00000102967 at 0x29cc5c990>, <Gene ENSG00000084774 at 0x29cc5cd50>}
{<Metabolite dhor_S_c at 0x29b6aaa10>, <Metabolite atp_c at 0x29b600850>, <Metabolite hco3_c at 0x29b67f850>, <Metabolite cbp_c at 0x29b6ed050>, <Metabolite pi_c at 0x29cc07e50>, <Metabolite glu_L_c at 0x29e40a690>, <Metabolite adp_c at 0x29b68b1d0>, <Metabolite orot_c at 0x29c2936d0>, <Metabolite h_c at 0x29e4adb10>, <Metabolite cbasp_c at 0x29b6eff10>, <Metabolite h2o_c at 0x29c2a7b50>, <Metabolite asp_L_c at 0x29b603350>, <Metabolite q10h2_m at 0x29cc05550>, <Metabolite q10_m at 0x29cc55d90>, <Metabolite gln_L_c at 0x29c2a7dd0>}


# Check the new metabolites, genes that were not in MitoCore and their annotations

 keep track for report: they dont have any annotations, need to be added in the alignment to Human1


In [119]:
for metabolite in metabolites_mitomammal:
     #check if metabolites are not in MitoCore (couple of metabolites shouldn't be there because they have not been added yet)
    if metabolite not in mitocore.metabolites:
        print(f"metabolite.id: {metabolite.id} annotation {metabolite.annotation}, formula {metabolite.formula}")

for gene in genes_mitomammal:
    #check if genes are not in MitoCore (some genes shouldn't be there such as mouse specific genes (ENSMUSG00000013629))
    if gene not in mitocore.genes:
        print(f"gene.id: {gene.id}", gene.annotation)

metabolite.id: dhor_S_c annotation {}, formula CH2NO5P
metabolite.id: cbp_c annotation {}, formula CH2NO5P
metabolite.id: orot_c annotation {}, formula CH2NO5P
metabolite.id: cbasp_c annotation {}, formula CH2NO5P
gene.id: ENSMUSG00000013629 {}
gene.id: ENSMUSG00000031730 {}
gene.id: ENSG00000102967 {}


# Add new features from MitoMAMMAL into MitoCore:

1. reactions
2. genes
3. metabolites

check again and remove murine genes such as ENSMUSG00000031730 from GPR and gene entries

In [120]:
from cobra import Model, Reaction, Metabolite

model = mitocore 
# 1. add reactions from mitomammal to mitocore
for reaction in mitomammal.reactions:
    if reaction.id in ['CBPS', 'ASPCT', 'DHORTS', 'DHORD9', 'DM_orot_c']:
        print(reaction.id)
        model.add_reactions([reaction])

# 2. add genes and metabolites from mitomammal to mitocore
for gene in genes_mitomammal:
    if gene not in mitocore.genes:
        print(gene.id)
        model.genes.append(gene)

# 3. add metabolites from mitomammal to mitocore
for metabolite in metabolites_mitomammal:
    if metabolite not in mitocore.metabolites:
        print(metabolite.id)
        model.metabolites.append(metabolite)



CBPS
ASPCT
DHORTS
DHORD9
DM_orot_c


# Introduce the newly curated GPRs in MitoMAMMAL for Complex I and IV 
1. CI_mitoMap
2. CIV_mitoMap

In [121]:

for reaction in mitocore.reactions:
    # 1. update GPR for Complex I
    if reaction.id =='CI_MitoCore':
        gpr_old= reaction.gene_reaction_rule
        gpr_new= mitomammal.reactions.get_by_id('CI_mitoMap').gene_reaction_rule
        print(f"Old GPR: {gpr_old}")
        print(f"New GPR: {gpr_new}")
        reaction.gene_reaction_rule = gpr_new
        print(f"Updated GPR for reaction {reaction.id}: {reaction.gene_reaction_rule}")
    # 2. update GPR for Complex IV
    if reaction.id =='CIV_MitoCore':
        gpr_old= reaction.gene_reaction_rule
        gpr_new= mitomammal.reactions.get_by_id('CIV_mitoMap').gene_reaction_rule
        print(f"Old GPR: {gpr_old}")
        print(f"New GPR: {gpr_new}")
        reaction.gene_reaction_rule = gpr_new
        print(f"Updated GPR for reaction {reaction.id}: {reaction.gene_reaction_rule}")

Old GPR: ENSG00000115286 and ENSG00000110717 and ENSG00000178127 and ENSG00000213619 and ENSG00000158864 and ENSG00000167792 and ENSG00000023228 and ENSG00000198888 and ENSG00000198763 and ENSG00000198840 and ENSG00000198886 and ENSG00000212907 and ENSG00000198786 and ENSG00000198695 and ENSG00000145494 and ENSG00000184752 and ENSG00000164258 and ENSG00000139180 and ENSG00000004779 and ENSG00000131495 and ENSG00000119013 and ENSG00000128609 and ENSG00000184983 and ENSG00000174886 and ENSG00000147123 and ENSG00000168653 and ENSG00000065518 and ENSG00000186010 and ENSG00000099795 and ENSG00000119421 and ENSG00000147684 and ENSG00000140990 and ENSG00000166136 and ENSG00000151366 and ENSG00000090266 and ENSG00000267855 and ENSG00000170906 and ENSG00000189043 and ENSG00000136521 and ENSG00000183648 and ENSG00000109390 and ENSG00000130414 and ENSG00000185633 and ENSG00000160194 and ENSG00000165264
New GPR: (ENSMUSG00000020153 and ENSMUSG00000059734 and ENSMUSG00000024099 and ENSMUSG000000055

In [122]:
# check added reactions
for reaction in mitocore.reactions:
    if reaction.id in ['CBPS', 'ASPCT', 'DHORTS', 'DHORD9', 'DM_orot_c']:
        annotations = reaction.annotation
        notes= reaction.notes
        metabolites= reaction.metabolites
        bounds= (reaction.lower_bound, reaction.upper_bound)
        print(f"reaction: {reaction.id}, GPR: {reaction.gene_reaction_rule}, annotations: {annotations}")
        print("--------------------------------------------------")


reaction: CBPS, GPR: ENSMUSG00000013629 or ENSG00000084774, annotations: {'sbo': 'SBO:0000176', 'bigg.reaction': 'CBPS', 'biocyc': 'META:CARBPSYN-RXN', 'ec-code': '6.3.5.5', 'kegg.reaction': 'R00575', 'metanetx.reaction': 'MNXR96485', 'reactome.reaction': ['R-RNO-73577', 'R-GGA-73577', 'R-DRE-73577', 'R-ATH-73577', 'R-CFA-73577', 'R-MMU-73577', 'R-OSA-73577', 'R-CEL-73577', 'R-SPO-73577', 'R-GGA-419440', 'R-BTA-73577', 'R-SCE-73577', 'R-XTR-73577', 'R-DDI-73577', 'R-HSA-73577', 'R-PFA-73577', 'R-TGU-73577', 'R-SSC-73577', 'R-DME-73577'], 'rhea': ['18635', '18633', '18636', '18634'], 'sabiork': '203', 'seed.reaction': 'rxn00414'}
--------------------------------------------------
reaction: ASPCT, GPR: ENSMUSG00000013629 or ENSG00000084774, annotations: {'sbo': 'SBO:0000176', 'bigg.reaction': 'ASPCT', 'biocyc': 'META:ASPCARBTRANS-RXN', 'ec-code': '2.1.3.2', 'kegg.reaction': 'R01397', 'metanetx.reaction': 'MNXR96080', 'reactome.reaction': ['R-DDI-73573', 'R-RNO-73573', 'R-SPO-73573', 'R-D

In [123]:
#check after adding genes and metabolites from MitoMAMMAL

for metabolite in metabolites_mitomammal:
    #check if metabolites are not in MitoCore (they should be there now after adding them)
    if metabolite not in mitocore.metabolites:
        print(f"metabolite.id: {metabolite.id} annotation {metabolite.annotation}, formula {metabolite.formula}")


for gene in genes_mitomammal:
    #check if genes are not in MitoCore (they should be there now after adding them)
    if gene not in mitocore.genes:
        print(f"gene.id: {gene.id}", gene.annotation)

# Clean genes and GPR from mouse specific genes (added from MitoMAMMAL)


In [124]:
def clean_not_human_genes(model):
    '''Remove non-human genes from the model'''
    non_human_genes = [gene for gene in model.genes if not gene.id.startswith("ENSG")]
    cobra.manipulation.delete.remove_genes(model, non_human_genes, remove_reactions=False)
    if non_human_genes:
        print(f"Removed non-human genes: {[gene.id for gene in non_human_genes]}")    
    return model


In [125]:
mitocore= clean_not_human_genes(mitocore)

Removed non-human genes: ['ENSMUSG00000013629', 'ENSMUSG00000031730', 'ENSMUSG00000035674', 'ENSMUSG00000064363', 'ENSMUSG00000029632', 'ENSMUSG00000037152', 'ENSMUSG00000022820', 'ENSMUSG00000002379', 'ENSMUSG00000025204', 'ENSMUSG00000031059', 'ENSMUSG00000113902', 'ENSMUSG00000024038', 'ENSMUSG00000064345', 'ENSMUSG00000024099', 'ENSMUSG00000022450', 'ENSMUSG00000059734', 'ENSMUSG00000027673', 'ENSMUSG00000064367', 'ENSMUSG00000033938', 'ENSMUSG00000014294', 'ENSMUSG00000037916', 'ENSMUSG00000026895', 'ENSMUSG00000013593', 'ENSMUSG00000041881', 'ENSMUSG00000020153', 'ENSMUSG00000040280', 'ENSMUSG00000026032', 'ENSMUSG00000022354', 'ENSMUSG00000026260', 'ENSMUSG00000061633', 'ENSMUSG00000020022', 'ENSMUSG00000021764', 'ENSMUSG00000000399', 'ENSMUSG00000028648', 'ENSMUSG00000030869', 'ENSMUSG00000064341', 'ENSMUSG00000065947', 'ENSMUSG00000071014', 'ENSMUSG00000064368', 'ENSMUSG00000040048', 'ENSMUSG00000030647', 'ENSMUSG00000025968', 'ENSMUSG00000002416', 'ENSMUSG00000005510', 'ENSMU

In [126]:
reaction = mitocore.reactions.get_by_id("CBPS")
print('original GPR:', reaction.gene_reaction_rule)


gpr_genes = split_gene_rule(reaction.gene_reaction_rule)
print('gpr splitted', gpr_genes)
new=" or ".join(gpr_genes)
print('new GPR:', new)

for reaction in mitocore.reactions:
    if reaction.id == "CIV_MitoCore":
        print(reaction.gene_reaction_rule)

original GPR: ENSG00000084774
gpr splitted ['ENSG00000084774']
new GPR: ENSG00000084774
ENSG00000198804 and ENSG00000198712 and ENSG00000198938 and (ENSG00000131143 or ENSG00000131055) and ENSG00000178741 and ENSG00000135940 and (ENSG00000111775 or ENSG00000156885) and (ENSG00000126267 or ENSG00000160471) and ENSG00000164919 and (ENSG00000161281 or ENSG00000112695 or ENSG00000115944) and ENSG00000131174 and ENSG00000127184 and (ENSG00000176340 or ENSG00000187581)


In [127]:
# load and clean model from N/A values
model_clean = clean_invalid_annotations(mitocore)
print(type(model_clean))
# save cleaned model as new SBML file
cobra.io.write_sbml_model(model_clean, "/Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/MitoCore_Curation/Mitocore_MitoMAMMAL.xml")
# Memote report after cleaning model to check if positive Memote features remain the same or false positive were present
!memote report snapshot --filename "/Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/MitoCore_Curation/Mitocore_MitoMAMMAL.html" /Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/MitoCore_Curation/Mitocore_MitoMAMMAL.xml

<class 'cobra.core.model.Model'>
The current solver interface glpk doesn't support setting the optimality tolerance.
============================= test session starts ==============================
platform darwin -- Python 3.10.8, pytest-7.1.2, pluggy-1.0.0
rootdir: /Users/benjaminreyes
plugins: anyio-3.5.0, typeguard-4.4.2
collected 155 items / 1 skipped                                                

../../../../../../anaconda3/lib/python3.10/site-packages/memote/suite/tests/test_annotation.py F [  0%]
FFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFF.FFFFFFF.F.....F.F.F.FFFFFFFF..         [ 41%]
../../../../../../anaconda3/lib/python3.10/site-packages/memote/suite/tests/test_basic.py . [ 42%]
.....FF.......F.F.F.FF                                                   [ 56%]
../../../../../../anaconda3/lib/python3.10/site-packages/memote/suite/tests/test_biomass.py . [ 57%]
FFFFFF....FFFFF.FF                                                       [ 69%]
../../../../../../anaconda3/lib/python3.10/site-

# -----------------Alignment of MitoCore to Human1---------------------

Align MitoCore annotations to Human1 

# Workflow Functions to:
1. Extract ids from model  
2. Map them to a target database in a mapping file
3. Add them to current model
4. Get attribute (not inside of attribute 'annotation', e.g. GPR rule)
5. Add GPR rule to the model and complete if already present.

In [128]:
# 1. extract ids from model  
def get_model_ids_for_databases(model, component_type, dict, databases):
    """
    Extract ids from model annotations for various databases across a specified component type.
    
    Parameters:
    - model: COBRApy model object (mitocore)
    - component_type: str, one of 'reactions', 'metabolites', 'genes'
    - dict: dictionary where database ids are located ('annotations', 'notes')
    - databases: list of str, e.g., ['kegg', 'bigg', 'metanetx', 'uniprot'].
    
    Returns:
    - DataFrame with component ID and extracted database ids.
    """
    # Initialize dictionary
    ids_dict = {'model_id': []}
    for db in databases: 
        ids_dict[db] = []

    # Access model core-component (reactions, genes, metabolites)
    components = getattr(model, component_type)

    for comp in components:
        ids_dict['model_id'].append(comp.id)

        for db in databases:
            value = None
            annotation = getattr(comp, dict, {})

            # look for different naming conventions
            for key in [db, 
                        f"{db}.reaction", 
                        f"{db}.compound", 
                        f"{db}.metabolite", 
                        f"{db}.chemical", 
                        f"{db}.genes",
                        f"{db}.gene",
                        f"{db} id",
                        f"{db.upper()} id",
                        ]:
                if key in annotation:
                    value = annotation[key]
                    break

            ids_dict[db].append(value)

    return pd.DataFrame(ids_dict)

# 2. map them to a target database in a mapping file
def map_ids_to_db(id_df, input_df, id_col_input, id_col_map, target_db_col, sep_map_file):
    """
    Map ids from a column in a dataframe to another column of a dataframe and add the target identifier.
    
    Parameters:
    - id_df: input Data frame with model_ids and ids to a specific database to be mapped.
    -input_df: dataframe for mapping ids to the same database as id_df and target database
    -id_col_input: column in id_df that contains ids to be mapped.
    -id_col_map: the column in mapping_file that contains the ids to be mapped.
    -target_db_col: the column in mapping_file that contains the target database ids to be extracted.
    -sep_map_file: the separator used in the mapping file (default is ',').
    
    Returns:
    - DataFrame with model ids and extracted target database ids.
    """
    import pandas as pd
    ids_dict = {'model_id': [],target_db_col: []}

    #load csv mapping file
    mapping_df = input_df

    # iterate over ids 
    for index, row in id_df.iterrows():
        id = row[id_col_input]

        # check for model ids are present in mapping file 
        match= mapping_df[mapping_df[id_col_map] == id]
        # if id model existing extract target id (target database)
        if not match.empty:
            target_id= match[target_db_col].values[0]
            print(f"Match found: {id} -> {target_id}")
        # if id model not found extract 'None' value
        else:
            target_id= None
            print(f"No match found for {id}, using default value None")

        #collect target ids into ids dictionary
        ids_dict['model_id'].append(row['model_id'])
        ids_dict[target_db_col].append(target_id)
    
    #return dictionary as dataframe
    ids_dict= pd.DataFrame(ids_dict)
    return ids_dict

# 3. add ids to model to the annotations
def add_ids_to_model(model, df, component_type, database):
    """
    Add ids from a DataFrame to the model annotations for a specified component type.
    new defined in automation pipeline to handle not stringtified lists in dataframes insted of .csv stringtified lists 

    Parameters:
    - model: COBRApy model object (e.g., mitocore)
    - df: DataFrame with model_id and database id columns.
    - component_type: str, one of 'reactions', 'metabolites', 'genes'
    - database: str, the name of the database column in df to map (e.g., 'kegg', 'bigg')
 
    
    Returns:
    - Updated model with new annotations added (only if not already present).
    """
    import ast
    components = getattr(model, component_type)

    for index, row in df.iterrows():
        model_id = row['model_id']
        db_value = row[database]

        # skip missing or None values 
        if db_value is None or (isinstance(db_value, float) and pd.isna(db_value)):
            continue

        # convert stringified lists (from CSVs) into real lists
        if isinstance(db_value, str) and db_value.startswith("[") and db_value.endswith("]"):
            try:
                db_value = ast.literal_eval(db_value)
            except Exception as e:
                print(f"Error parsing {db_value}: {e}")
                continue

        # now db_value is either a string or a real list
        for comp in components:
            if comp.id == model_id:
                if isinstance(db_value, list) and len(db_value) == 1:
                    db_value = db_value[0]
                comp.annotation[database] = db_value
                print(f"Added {db_value} (type {type(db_value)}) to {model_id}")
                break

    return model

# 4. get attribute (not inside of attribute 'annotation', e.g.GPR rule)
def get_model_ids_for_attr(model, component_type, attr_name):
    """
    Extract attribute in the model in the target core-component.
    
    Parameters:
    - model: COBRApy model object (mitocore)
    - component_type: Core component of GEMs, str, one of 'reactions', 'metabolites', 'genes'
    - attr_name: name of the dictionary where target object is located ('notes', '_gpr')

    
    Returns:
    - DataFrame with component ID and extracted database ids.
    """
    # dictionary to collect ids and attributes
    ids_dict = {'model_id': [], attr_name:[]}

    # Access model component (e.g., reactions, genes)
    components = getattr(model, component_type)

    # iterate over components and extract attribute
    for comp in components:
        #add model id and attribute to dictionary
        ids_dict['model_id'].append(comp.id)
        value = comp.__dict__.get(attr_name, None) #None in case no attribute present
        ids_dict[attr_name].append(value) 

    #returns dataframe with attributes
    return pd.DataFrame(ids_dict)

# 6. add ids to model notes (not inside of 'annotation' attribute)
def add_ids_to_model_notes(model, df, component_type, database):
    """
    Add ids from a DataFrame to the model annotations for a specified component type.
    new defined in automation pipeline to handle not stringtified lists in dataframes insted of .csv stringtified lists 

    Parameters:
    - model: COBRApy model object (e.g., mitocore)
    - df: DataFrame with model_id and database id columns.
    - component_type: str, one of 'reactions', 'metabolites', 'genes'
    - database: str, the name of the database column in df to map (e.g., 'kegg', 'bigg')
 
    
    Returns:
    - Updated model with new annotations added (only if not already present).
    """
    import ast
    components = getattr(model, component_type)

    for index, row in df.iterrows():
        model_id = row['model_id']
        db_value = row[database]

        # skip missing or None values 
        if db_value is None or (isinstance(db_value, float) and pd.isna(db_value)):
            continue

        # convert stringified lists (from CSVs) into real lists
        if isinstance(db_value, str) and db_value.startswith("[") and db_value.endswith("]"):
            try:
                db_value = ast.literal_eval(db_value)
            except Exception as e:
                print(f"Error parsing {db_value}: {e}")
                continue

        # now db_value is either a string or a real list
        for comp in components:
            if comp.id == model_id:
                if isinstance(db_value, list) and len(db_value) == 1:
                    db_value = db_value[0]
                comp.notes[database] = db_value
                print(f"Added {db_value} (type {type(db_value)}) to {model_id}")
                break

    return model


In [129]:
# --Function to clean up N/A, None, nan... Values---
def clean_invalid_annotations(model, invalid_values={'N/A', 'None', 'nan', 'NaN', ''}):
    """
    Remove invalid placeholder values from model annotations across reactions, metabolites, and genes to avoid false positives in MEMOTE test.
    
    -model: COBRApy model object (already read with COBRApy, e.g. Mitocore)
    -invalid_values: dict, keys to be removed
    """
    #acces model's objects core-components 
    for component in model.reactions + model.metabolites + model.genes:
        #collect keys to remove
        keys_to_remove = []
        for key, val in component.annotation.items():
            if isinstance(val, (str, type(None))):
                #check if value inside key is in invalid dict
                if str(val).strip() in invalid_values:
                    keys_to_remove.append(key)
        for key in keys_to_remove:
            del component.annotation[key]
    # return model with changes (does not save model as new SBML file)
    return model

# MetaNetx file processing functions

There are two informations necessary from MetaNetX file:
1. MetaNetX identifier
2. ec-code

In [130]:
# 1.MetaNetX identifier
def mnx_processing_file_mnx(mnx_file,db_map,component,start_patterns):
    """Process the MNX mapping file to extract relevant columns and save it as a CSV.

    Parameters:
    - mnx_file: metanetx _xref.tsv file to process
    - db_map: reference ids to map for ['bigg', 'kegg'...]
    - component: either 'reactions' or 'metabolites' can be tracked in metanetx database.
    - start_patterns: Tupple of patterns to look for in file for id processing:
        e.g. ( "kegg.compound:", "keggC:") or ("bigg.metabolite:", "biggM:")
    
    Returns:
    - DataFrame with component id and extracted database ids.

    """ 
    #dictionary to collect database ids and respective metanetx ids
    mapping = {db_map: [], f"metanetx.{component}": []}

    #open and read metanetx mapping file
    with open(mnx_file, "r", encoding="utf-8") as file:
        # iterate over lines in file
        for line in file:
            #check for comments (#) and empty lines
            if line.startswith('#') or not line.strip():
                continue
            
            # split line by tab and check if has at least 3 columns
            parts = line.strip().split('\t')
            if len(parts) < 3:
                continue  # Skip malformed lines
            
            # mnx id is in 2°nd part out of 3 parts
            source = parts[0]
            target_mnx_id = parts[1]
            rest = parts[2]

            #generalize starting patterns to look for   KEGG or BIGG
            if source.startswith(start_patterns):
                #recon_id is a generic name for database id
                recon_id = source.split(":", 1)[1].strip()
                mnx_id = target_mnx_id.strip()
                mapping[db_map].append(recon_id)
                mapping[f"metanetx.{component}"].append(mnx_id)

    #return dataframe of collecting dictionary
    metanetx_react = pd.DataFrame.from_dict(mapping, orient="columns")
    return metanetx_react
# 2.ec-code
def mnx_processing_file_eccode(mnx_file, db_map):
    """Process the MNX mapping file to extract ec-codes and database ids.

    Parameters:
    - mnx_file: Path to the MetaNetX file with reaction or metabolite data
    - db_map: Database name (e.g., 'bigg', 'kegg') to store as a column

    Returns:
    - DataFrame with db_map and EC code columns (EC is a list if >1, else string)
    """
    # dictionary to collect database ids and respective ec-code
    mapping = {db_map: [], 'ec-code': []}
    # Regular expression pattern to match ec-code
    ec_pattern = re.compile(r"\b\d+\.\d+\.\d+(?:\.\d+|\.n)?\b")
    
    #open and read metanetx mapping file
    with open(mnx_file, "r", encoding="utf-8") as file:
        # iterate over lines in file
        for line in file:
            if line.startswith('#') or not line.strip():
                continue

            # split line by tab and check if has at least 4 columns
            parts = line.strip().split('\t')
            if len(parts) < 4:
                continue  # Skip malformed lines

            # ec-code is in 4°nd part out of 4 parts
            mnx_id = parts[0].strip()
            ec_codes = parts[3].strip()


            if ec_pattern.search(ec_codes):
                codes_patter= re.compile(r'^\d+\.\d+\.\d+\.\d+$')
                codes = [code.strip() for code in ec_codes.split(';') if code.strip()]
                codes = [code for code in codes if codes_patter.match(code)]
                # Convert to string if only one ec-code
                formatted_ec = codes[0] if len(codes) == 1 else codes
                mapping[db_map].append(mnx_id)
                mapping['ec-code'].append(formatted_ec)
                print(f"{formatted_ec} is  {type(formatted_ec)}")

    # return dataframe of collecting dictionary for ec-codes and respective mnx ids
    df = pd.DataFrame(mapping)
    return df

# Load Models
- MitoCore
- Human1

In [131]:
mitocore= cobra.io.read_sbml_model("Mitocore_MitoMAMMAL.xml")
human1= cobra.io.read_sbml_model('/Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/Human-GEM.xml')

# ----------------------------------REACTIONS CHECKPOINT-----------------------------------------


# Add MetaNetX ids to model

Worflow:
1. Process MNX tsv file (function 1.) and save to csv
2. Get ids from the model [KEGG, BiGG]
3. Map model ids to metanetx processed  csv file using [BiGG] --> Why not using KEGG too??
4. Add MNX ids to the model


In [132]:

# 2. Process mnx file (MNX processing function 1.)
mnx_reactions= mnx_processing_file_mnx('/Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/Files_Databases/MNX_reac_xref.tsv','Recon2','reaction',("bigg.reaction:", "biggR:"))
print('processes MNX reaction file')
print(mnx_reactions)

# 3. get kegg and bigg ids from model
print('geting ids for all possible databases ')
df_react = get_model_ids_for_databases(mitocore, 'reactions', 'notes', ['KEGG', 'Recon2'])
print(df_react)

# 4. map model ids (bigg) to metanetx processed  csv file
print('Mapping  metabolites to metanetx.')
df_mnx_react = map_ids_to_db(df_react, mnx_reactions,'Recon2', 'Recon2', 'metanetx.reaction', ',')
print('MNX metabolites \n',df_mnx_react)

# 5. Add mnx ids to the model
print('adding MNX ids to the model')
mnx_react = add_ids_to_model(mitocore, df_mnx_react, 'reactions','metanetx.reaction', )




processes MNX reaction file
             Recon2 metanetx.reaction
0            CRBNTD             EMPTY
1          DNADRAIN             EMPTY
2           H2CO3D2             EMPTY
3          H2CO3D2m             EMPTY
4          HMR_5409             EMPTY
...             ...               ...
112679  R_GALNACT3g         MNXR99998
112680    GALNACT4g         MNXR99999
112681  R_GALNACT4g         MNXR99999
112682    GALNACT4g         MNXR99999
112683  R_GALNACT4g         MNXR99999

[112684 rows x 2 columns]
geting ids for all possible databases 
       model_id  KEGG Recon2
0      EX_2hb_e  None   None
1       EX_ac_e  None   None
2     EX_acac_e  None   None
3      EX_akg_e  None   None
4    EX_ala_B_e  None   None
..          ...   ...    ...
555        CBPS  None   None
556       ASPCT  None   None
557      DHORTS  None   None
558      DHORD9  None   None
559   DM_orot_c  None   None

[560 rows x 3 columns]
Mapping  metabolites to metanetx.
No match found for None, using default value

# Add new Human1 ids to model 
### based on upgraded  MetaNetX ids 
- MetaNetX is the only id updated during CI so no need to test for KEGG or BiGG 

Worflow:
1. Load last SBML model and extract database ids into dataframe
2. Map reactions to Human1 in .tsv file and rename naming for Human1
3. Add Human1 ids from dataframe to model.


In [133]:
#-----------BiGG to Human1 mapping and adding Human1 ids to the model-----------------
print('geting ids for all bigg')
df_react = get_model_ids_for_databases(mitocore, 'reactions', '_annotation', ['bigg'])
print(df_react)

# map bigg ids to Human1 mapping file
human1_react= pd.read_csv('/Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/Files_Databases/Human1_reactions.tsv', sep='\t')
df_human1_react = map_ids_to_db(df_react, human1_react,'bigg', 'rxnBiGGID', 'rxns', '\t')
print('human1 reactions \n',df_human1_react)
df_human1_react = df_human1_react.rename(columns={'model_id': 'model_id', 'rxns': 'Human1'})
print(df_human1_react)

# add Human1 ids to model
print('adding human1 ids to the model notes')
mitocore = add_ids_to_model_notes(mitocore, df_human1_react, 'reactions','Human1')

geting ids for all bigg
       model_id    bigg
0      EX_2hb_e    None
1       EX_ac_e    None
2     EX_acac_e    None
3      EX_akg_e    None
4    EX_ala_B_e    None
..          ...     ...
555        CBPS    CBPS
556       ASPCT   ASPCT
557      DHORTS  DHORTS
558      DHORD9  DHORD9
559   DM_orot_c    None

[560 rows x 2 columns]
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found

In [134]:
#-----------KEGG to Human1 mapping and adding Human1 ids to the model-----------------


print('geting ids for all kegg')
df_react = get_model_ids_for_databases(mitocore, 'reactions', '_annotation', ['kegg'])
print(df_react)
df_react = df_react.explode('kegg')
print(df_react)

# 2. map bigg ids to Human1 mapping file
print('Mapping kegg reactions to human1')
df_human1_react = map_ids_to_db(df_react, human1_react,'kegg', 'rxnKEGGID', 'rxns', '\t')
print('human1 reactions \n',df_human1_react)
df_human1_react = df_human1_react.rename(columns={'model_id': 'model_id', 'rxns': 'Human1'})
print(df_human1_react)

# 3. add Human1 ids to model
print('adding human1 ids to the model')
mitocore = add_ids_to_model_notes(mitocore, df_human1_react, 'reactions','Human1')

geting ids for all kegg
       model_id    kegg
0      EX_2hb_e    None
1       EX_ac_e    None
2     EX_acac_e    None
3      EX_akg_e    None
4    EX_ala_B_e    None
..          ...     ...
555        CBPS  R00575
556       ASPCT  R01397
557      DHORTS  R01993
558      DHORD9    None
559   DM_orot_c    None

[560 rows x 2 columns]
       model_id    kegg
0      EX_2hb_e    None
1       EX_ac_e    None
2     EX_acac_e    None
3      EX_akg_e    None
4    EX_ala_B_e    None
..          ...     ...
555        CBPS  R00575
556       ASPCT  R01397
557      DHORTS  R01993
558      DHORD9    None
559   DM_orot_c    None

[596 rows x 2 columns]
Mapping kegg reactions to human1
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found for 

In [135]:
#-----------MetaNetX to Human1 mapping and adding Human1 ids to the model-----------------
print('geting ids for all MNX')
df_react = get_model_ids_for_databases(mitocore, 'reactions', '_annotation', ['metanetx'])
print(df_react)
# metanetx ids to Human1 mapping file
print('Mapping mnx reactions to human1')
df_human1_react = map_ids_to_db(df_react, human1_react,'metanetx', 'rxnMetaNetXID', 'rxns', '\t')
print('human1 reactions \n',df_human1_react)
df_human1_react = df_human1_react.rename(columns={'model_id': 'model_id', 'rxns': 'Human1'})
print(df_human1_react)
# add Human1 ids to model
print('adding human1 ids to the model')
mitocore = add_ids_to_model_notes(mitocore, df_human1_react, 'reactions','Human1')



geting ids for all MNX
       model_id   metanetx
0      EX_2hb_e       None
1       EX_ac_e       None
2     EX_acac_e       None
3      EX_akg_e       None
4    EX_ala_B_e       None
..          ...        ...
555        CBPS  MNXR96485
556       ASPCT  MNXR96080
557      DHORTS  MNXR97428
558      DHORD9  MNXR97423
559   DM_orot_c       None

[560 rows x 2 columns]
Mapping mnx reactions to human1
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value None
No match found for None, using default value No

# --------Checkpoint MetaNetX, Human1 ids updated----

In [136]:

def visualization_gpr(model, name, timeline):
    """ function to extract number of reactions with GPR and without GPR in a metabolic SBML model
    
    -model: str, path to the SBML model
    -name: str , name of the model
    -timeline: str, stand of the model for naming (e.g. prior, after)
    """

    model
    has_gpr=[]
    no_gpr=[]
    
    for reaction in model.reactions:
        if reaction.gene_reaction_rule.strip():
            has_gpr.append(reaction)
        else:
            no_gpr.append(reaction)

    print(f"{name} reactions with GPR {timeline}:", len(has_gpr))
    print(f"{name}_reactions empty GPR {timeline}:", len(no_gpr))

In [137]:
## --Visualization of GPR rules in Mitocore prior--
mito=visualization_gpr(mitocore,'mitocore', 'prior')

mitocore reactions with GPR prior: 357
mitocore_reactions empty GPR prior: 203


# Align Mitocore GPR list to Human1

Worflow:

2. Extract Human1 ids from Mitocore into dataframe.
3. Extract GPRs from models
4. Merge Dataframes 
5. Parse GPRs present in both models
6. Add GPRs (risk of duplicates, cleaned later) except for list of MitoMAMMAL manually curated GPRs ['CBPS', 'ASPCT', 'DHORTS', 'DHORD9', 'DM_orot_c', 'CI_MitoCore', 'CIV_MitoCore'] 


In [138]:
def get_model_ids_for_attr(model, component_type, attr_name):

    ids_dict = {'model_id': [], attr_name: []}

    components = getattr(model, component_type)

    for comp in components:
        ids_dict['model_id'].append(comp.id)

        # SPECIAL CASE FOR GPR
        if attr_name in ['gpr', 'gene_reaction_rule', '_gpr']:
            value = comp.gene_reaction_rule
        else:
            value = getattr(comp, attr_name, None)

        ids_dict[attr_name].append(value)

    return pd.DataFrame(ids_dict)

# 5. add GPR rule to the model and complete if already present
def add_gpr_to_model(model, df, component_type, gpr_header_in_df):
    """
    check if attribute GPR is empty in the model or already present,
    Add ids from a dataFrame to the gpr attribute if empty or append to existing GPR rule.

    Parameters:
    - model: COBRApy model object (e.g., mitocore)
    - df: DataFrame with model_id and formulas column.
    - component_type: str, one of 'reactions', 'metabolites', 'genes'
    - gpr_header_in_df: str, the name of the database column in df to map (e.g., '_gpr')
    
    Returns:
    - Updated model with new annotations added (only if not already present).
    """
    # Access model component (e.g., reactions, genes)
    components = getattr(model, component_type )

    # iterate over ids (rows)
    for index, row in df.iterrows():
        model_id = row['model_id']
        new_rule = str(row[gpr_header_in_df]).strip() if pd.notnull(row[gpr_header_in_df]) else ''

        # find matching component based on model_id
        for comp in components:
            if comp.id == model_id and model_id not in ['CBPS', 'ASPCT', 'DHORTS', 'DHORD9', 'DM_orot_c', 'CI_MitoCore', 'CIV_MitoCore']:  # Exclude newly added reactions
                existing_rule = comp.gene_reaction_rule.strip()
                #check if GPR is empty)
                if existing_rule == '' and new_rule != '':
                    comp.gene_reaction_rule = new_rule
                    print(f"Added GPR: {comp.id} -> {new_rule}")
                #check if GPR already exists and append new rule with 'or' (does not avoid duplicates, this issue is corrected later)
                elif existing_rule != '' and new_rule != '':
                    comp.gene_reaction_rule = existing_rule + ' or ' + new_rule
                    print(f"GPR already exists for {comp.id}: {existing_rule}. New rule {new_rule} added.")
                break  # Stop once matched
    return model


# -------------Reactions---------------

In [139]:

# 2. extract Human1 ids in mitocore
human1_mitocore=get_model_ids_for_databases(mitocore, 'reactions', 'notes', ['Human1'])
#print(human1_mitocore)
# 3. extract GPR rules from models
gpr_mitocore= get_model_ids_for_attr(mitocore, 'reactions', '_gpr')
#print(gpr_mitocore)
gpr_human1= get_model_ids_for_attr(human1, 'reactions', '_gpr')

print(gpr_human1)
# 4. merge mitocore dfs to have Human1 ids and GPR rules.
gpr_mitocore = gpr_mitocore.merge(human1_mitocore, on='model_id')
print(gpr_mitocore)
print('df with _gpr and model ids in Mitocore', gpr_mitocore)
# 5. df with gpr from human1 mapped to mitocore
gpr_parsed=map_ids_to_db(gpr_mitocore, gpr_human1, 'Human1', 'model_id', '_gpr', ',')
print(gpr_parsed)

# 6. add gpr from human1 to mitocore and Memote report after adding GPRs
gpr_add=add_gpr_to_model(mitocore, gpr_parsed, 'reactions', '_gpr')

       model_id                                               _gpr
0      MAR03905  ENSG00000147576 or ENSG00000172955 or ENSG0000...
1      MAR03907                                    ENSG00000117448
2      MAR04097                                    ENSG00000131069
3      MAR04099                 ENSG00000111058 or ENSG00000154930
4      MAR04108                                    ENSG00000131069
...         ...                                                ...
12966  MAR20179                                                   
12967  MAR20180                                                   
12968  MAR20181                                    ENSG00000125454
12969  MAR20182                                    ENSG00000125454
12970  MAR20183                                    ENSG00000125454

[12971 rows x 2 columns]
       model_id             _gpr    Human1
0      EX_2hb_e                       None
1       EX_ac_e                       None
2     EX_acac_e                       Non

In [140]:
mito=visualization_gpr(mitocore,'mitocore', 'after gpr Human1')


mitocore reactions with GPR after gpr Human1: 405
mitocore_reactions empty GPR after gpr Human1: 155


# Cleanning GPRs from duplicates 

the function to add GPRs from Human1 to Memote checks if the reactions has empty GPR and adds it directly. In case there is a GPR present in for the entry, it adds the whole Human1 GPR to Mitocore with 'or' partricle.

## Example:
### -Reaction R1 in Mitocore  --> GPR ['geneA'] 
### -Reaction R1 in Human1 --> GPR ['geneA' or 'geneB']

### after add new GPR to Mitocore:
### - New Reaction R1 in Mitocore --> GPR ['geneA' or 'geneA' or 'geneB']


# Eliminate duplicates and keep new elements added to GPR
GPR is constructed with 2 conectors for the genes:
#
- 'and' reflects a requirement, genes >1 represent a single element. Built in parenthesis ()
e.g: (geneA and geneB)
- 'or' spearates elements in the GPR, every gene separated by 'or' is a new element. No parenthesis 
e.g: geneA or geneB
# Worflow: 
1. split elements (separator 'or')
2. Collect GPR for every single reaction and eliminate duplicates
3. Split elements inside parenthesis (separator 'and') and randomize them with tuple
4. re-join the GPR by replacing ',' by 'and' and joining single elements with 'or'
5. Create a csv file from the dataframe for manual control.
6. Update GPRs in Mitocore, save new SBML model, and run MEMOTE test


In [141]:

#create df with uncleaned GPRs 
gpr_dict={'model_id':[], 'gpr':[]}

for reaction in mitocore.reactions:
    gpr=reaction.gene_reaction_rule
    if gpr:
        gpr_dict['model_id'].append(reaction.id)
        gpr_dict['gpr'].append(gpr)
    else:
        gpr_dict['model_id'].append(reaction.id)
        gpr_dict['gpr'].append('')

gpr_df = pd.DataFrame(gpr_dict)


# 1. Split GPR on 'or'
for index, row in gpr_df.iterrows():
    gpr = row['gpr']
    elements_in_gpr = str(gpr).split(' or ')

    # 2. remove duplicates for each reaction with function set()
    unique_parentheses = set() # colect single genes in parenthesis separated by 'and' (ENSG00000115361 and ENSG00000171503) --> ('ENSG00000115361', 'ENSG00000115361')
    unique_single_genes = set()
    cleaned_gprs = [] 
    pattern = r"\((.*?)\)"  # matches content inside parentheses

    # 3. iterate through each element in the GPR: (parenthesis or single gene) 
    for element in elements_in_gpr:
        element = element.strip()

        # genes in  a parehteses such as: (ENSG00000115361 and ENSG00000171503)
        if element.startswith('(') and 'and' in element:
            match = re.search(pattern, element)
            if match:
                genes = match.group(1).split(' and ')
                #randomize order with tuple: order-independent for comparisson
                genes = tuple(sorted(g.strip() for g in genes)) 
                if genes not in unique_parentheses:
                    # colect single genes in parenthesis separated by 'and' 
                    # (ENSG00000115361 and ENSG00000171503) --> ('ENSG00000115361', 'ENSG00000115361')
                    unique_parentheses.add(genes)
                    #join single genes back with separator 'and' in the cleaned_gprs list
                    cleaned_gprs.append(f"({' and '.join(genes)})")

        # Single gene (not in parentheses)
        elif element not in unique_single_genes:
            unique_single_genes.add(element)
            cleaned_gprs.append(element)

    # 4. update the GPR back to the DataFrame with ' or ' as separator
    gpr_df.at[index, 'gpr'] = ' or '.join(cleaned_gprs)



In [142]:
# 5. update new GPRs from df to the model, save and test model
for reaction in mitocore.reactions:
    id = reaction.id
    model_gpr = reaction.gene_reaction_rule

    for index, row in gpr_df.iterrows():
        if row['model_id'] == id:  # Use an if statement to check the condition
            new_gpr = row['gpr']  # Access the '_gpr' column of the current row
            print(f"Updating GPR for {model_gpr} to {new_gpr}")
            reaction.gene_reaction_rule = new_gpr
            break

Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating GPR for  to 
Updating G

In [143]:
mito=visualization_gpr(mitocore,'mitocore', 'after cleaning GPR')

mitocore reactions with GPR after cleaning GPR: 405
mitocore_reactions empty GPR after cleaning GPR: 155


In [144]:
# Split GPR on 'or' 
def gpr_counte( model):
    """ function to count the absolute number of GPR 
    splits GPR based on 'or'

    -df: DataFrame with GPRs and model ids 
    -model: str SBML model
    
    retunr: GPR count 
    """
    gpr_absolute= []

    for reaction in model.reactions:
        gpr= reaction.gene_reaction_rule
        elements_in_gpr = str(gpr).split(' or ')

        for element in elements_in_gpr:
            if element not in [None, 'N/A','']:
                gpr_absolute.append(element)
    print(gpr_absolute)
    return print(f" {model}, has {len(gpr_absolute)} GPRs" )

mitocore0 = cobra.io.read_sbml_model('/Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/MitoCore_Curation/Mitocore_MitoMAMMAL.xml')
gpr_0 = gpr_counte(mitocore0)
gpr_1 = gpr_counte(mitocore)

['ENSG00000156515', 'ENSG00000159399', 'ENSG00000160883', 'ENSG00000106633', 'ENSG00000131482', 'ENSG00000105220', 'ENSG00000152556', 'ENSG00000165140', 'ENSG00000130957', 'ENSG00000149925', 'ENSG00000109107', 'ENSG00000111669', 'ENSG00000111640', 'ENSG00000102144', 'ENSG00000171314', 'ENSG00000164708', 'ENSG00000074800', 'ENSG00000111674', 'ENSG00000108515', 'ENSG00000067225', 'ENSG00000067225', 'ENSG00000124253', 'ENSG00000111716', 'ENSG00000160211', 'ENSG00000130313', 'ENSG00000142657', 'ENSG00000153574', 'ENSG00000235376', 'ENSG00000163931', 'ENSG00000177156', 'ENSG00000163931', 'ENSG00000131828 and ENSG00000168291 and ENSG00000150768 and ENSG00000091140', 'ENSG00000062485', 'ENSG00000100412', 'ENSG00000166411 and ENSG00000101365 and ENSG00000067829', 'ENSG00000182054', 'ENSG00000105953 and ENSG00000119689 and ENSG00000091140', 'ENSG00000163541 and ENSG00000172340', 'ENSG00000136143', 'ENSG00000091483', 'ENSG00000146701', 'ENSG000001152863 and ENSG00000110717 and ENSG00000178127 an

# ---------Checkpoint GPRs updated----

# Update ec-code list in Mitocore

We asume Human1 is a well curated model and align the GPR list if possible to Human1. Do not preserve Mitocore ec-code unless there is no ec-code from Human1 to take over

Worflow:
1. Process MetaNetX file to find ec-codes and save as csv file 
2. Load latest SBML model
3. Extract MetaNetX ids from models
4. Map metanetx ids to ec-codes 
5. Overwrite the ec-codes for each reaction  with new ec-code list from Human1 and Memote test 


In [145]:
# 1.extract ec-codes and corresponding mnx id from metanetx file and create a csv file 
ec_code = mnx_processing_file_eccode('/Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/Files_Databases/reac_prop.tsv', 'metanetx.reaction')

# 3. extract metanetx ids from model
print('geting ids for all mnx reactions')
df_model_id = get_model_ids_for_databases(mitocore, 'reactions', '_annotation', ['metanetx.reaction'])
print(df_model_id)

# 4. map metanetx ids to ec-codes
print('Mapping  reactions')
df_react_map = map_ids_to_db(df_model_id, ec_code ,'metanetx.reaction', 'metanetx.reaction', 'ec-code', ',')
print('ec_code reactions \n',df_react_map)

# 5. add ec-codes to model and Memote test
print('adding ec-codes to the model')
mnx_react = add_ids_to_model(mitocore, df_react_map, 'reactions','ec-code')


6.3.1.2 is  <class 'str'>
['1.4.1.13', '1.4.1.14', '1.4.7.1', '2.4.2.14', '3.5.1.2', '3.5.1.38', '4.3.3.6', '6.3.4.2', '6.3.5.2', '6.3.5.4', '6.3.5.5', '6.3.5.7'] is  <class 'list'>
['1.2.1.19', '1.2.1.21', '1.2.1.3', '1.2.1.4', '1.2.1.5', '1.2.1.69'] is  <class 'list'>
5.5.1.19 is  <class 'str'>
2.1.2.10 is  <class 'str'>
1.4.4.2 is  <class 'str'>
['1.6.2.6', '1.8.1.4'] is  <class 'list'>
6.3.2.2 is  <class 'str'>
['1.2.1.3', '1.2.1.54'] is  <class 'list'>
['1.4.1.13', '1.4.1.3', '1.4.1.4'] is  <class 'list'>
1.8.1.7 is  <class 'str'>
['1.6.4.2', '1.8.1.7', '1.8.4.2'] is  <class 'list'>
['3.6.1.11', '3.6.1.40'] is  <class 'list'>
1.2.1.99 is  <class 'str'>
3.5.1.94 is  <class 'str'>
2.5.1.41 is  <class 'str'>
3.1.3.69 is  <class 'str'>
['2.3.2.2', '3.4.19.14'] is  <class 'list'>
[] is  <class 'list'>
['2.3.2.2', '3.4.19.13'] is  <class 'list'>
[] is  <class 'list'>
4.1.2.20 is  <class 'str'>
3.6.3.17 is  <class 'str'>
1.4.7.1 is  <class 'str'>
6.3.1.2 is  <class 'str'>
['3.6.3.21', '7

# Take over the Human1 annotations if not already in Mitocore

In [146]:
for reaction_hum in human1.reactions:
    human1_id = reaction_hum.id

    for reaction_mit in mitocore.reactions:
        notes = reaction_mit.notes

        # check if Human1 mapping exists
        if "Human1" not in notes:
            continue

        if notes["Human1"] != human1_id:
            continue

        # merge annotations: Human1 to MitoCore
        for ann_key, ann_val in reaction_hum.annotation.items():
            # only add if MitoCore does NOT already contain the annotation
            if ann_key not in reaction_mit.annotation:
                reaction_mit.annotation[ann_key] = ann_val

In [147]:
for react in mitocore.reactions:
    annotations=react.annotation
    print(react.id, annotations)

EX_2hb_e {'sbo': 'SBO:0000627'}
EX_ac_e {'sbo': 'SBO:0000627'}
EX_acac_e {'sbo': 'SBO:0000627'}
EX_akg_e {'sbo': 'SBO:0000627'}
EX_ala_B_e {'sbo': 'SBO:0000627'}
EX_ala_L_e {'sbo': 'SBO:0000627'}
EX_arg_L_e {'sbo': 'SBO:0000627'}
EX_argsuc_e {'sbo': 'SBO:0000627'}
EX_asn_L_e {'sbo': 'SBO:0000627'}
EX_asp_L_e {'sbo': 'SBO:0000627'}
EX_bhb_e {'sbo': 'SBO:0000627'}
EX_bilirub_e {'sbo': 'SBO:0000627'}
EX_biomass_e {'sbo': 'SBO:0000627'}
EX_but_e {'sbo': 'SBO:0000627'}
EX_chol_e {'sbo': 'SBO:0000627'}
EX_cit_e {'sbo': 'SBO:0000627'}
EX_citr_L_e {'sbo': 'SBO:0000627'}
EX_co_e {'sbo': 'SBO:0000627'}
EX_co2_e {'sbo': 'SBO:0000627'}
EX_creat_e {'sbo': 'SBO:0000627'}
EX_cyan_e {'sbo': 'SBO:0000627'}
EX_cys_L_e {'sbo': 'SBO:0000627'}
EX_etoh_e {'sbo': 'SBO:0000627'}
EX_fe2_e {'sbo': 'SBO:0000627'}
EX_for_e {'sbo': 'SBO:0000627'}
EX_fum_e {'sbo': 'SBO:0000627'}
EX_glc_D_e {'sbo': 'SBO:0000627'}
EX_gln_L_e {'sbo': 'SBO:0000627'}
EX_glu_L_e {'sbo': 'SBO:0000627'}
EX_gly_e {'sbo': 'SBO:0000627'}
EX_g

# ----------------Genes-----------------

In [148]:

human1= cobra.io.read_sbml_model('/Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/Human-GEM.xml')



Transfer:

1. Human1 ids --> notes 
2. ensembl ids --> annotations

In [149]:
#Human1 ids --> notes 
Human1_ids=[]

#loop over genes in mitocore
for gene in mitocore.genes:
    #get Human1 id if existing otherwise 'None' entry
    human1_id= gene.notes.get('Human1', None)
    # use gene id as Human1 id if no Human1 id existing
    id=gene.id
    Human1_ids.append(id)

    if human1_id is None:
        gene.notes['Human1'] = id

#ensembl ids --> the annotations
ensembl_ids=[]
#loop over genes in mitocore
for gene in mitocore.genes:
    #get Human1 id if existing otherwise 'None' entry
    ensembl_id= gene.annotation.get('ensembl', None)
    # use gene id as Human1 id if no Human1 id existing
    id=gene.id
    ensembl_ids.append(id)

    if human1_id is None:
        gene.annotation['ensembl'] = id

In [150]:
#check the notes
for react in mitocore.genes:
    notes=react.notes
    print(react.id, notes)

ENSG00000159399 {'Human1': 'ENSG00000159399'}
ENSG00000106633 {'Human1': 'ENSG00000106633'}
ENSG00000156515 {'Human1': 'ENSG00000156515'}
ENSG00000160883 {'Human1': 'ENSG00000160883'}
ENSG00000131482 {'Human1': 'ENSG00000131482'}
ENSG00000105220 {'Human1': 'ENSG00000105220'}
ENSG00000152556 {'Human1': 'ENSG00000152556'}
ENSG00000165140 {'Human1': 'ENSG00000165140'}
ENSG00000130957 {'Human1': 'ENSG00000130957'}
ENSG00000109107 {'Human1': 'ENSG00000109107'}
ENSG00000149925 {'Human1': 'ENSG00000149925'}
ENSG00000111669 {'Human1': 'ENSG00000111669'}
ENSG00000111640 {'Human1': 'ENSG00000111640'}
ENSG00000102144 {'Human1': 'ENSG00000102144'}
ENSG00000164708 {'Human1': 'ENSG00000164708'}
ENSG00000171314 {'Human1': 'ENSG00000171314'}
ENSG00000108515 {'Human1': 'ENSG00000108515'}
ENSG00000111674 {'Human1': 'ENSG00000111674'}
ENSG00000074800 {'Human1': 'ENSG00000074800'}
ENSG00000067225 {'Human1': 'ENSG00000067225'}
ENSG00000124253 {'Human1': 'ENSG00000124253'}
ENSG00000111716 {'Human1': 'ENSG00

In [151]:
#check the genes
for react in mitocore.genes:
    notes=react.annotation
    print(react.id, notes)

ENSG00000159399 {'hgnc': 'HGNC:4923', 'hgnc.symbol': 'HK2', 'ensembl': 'ENSG00000159399', 'uniprot': 'P52789'}
ENSG00000106633 {'hgnc': 'HGNC:4195', 'hgnc.symbol': 'GCK', 'ensembl': 'ENSG00000106633', 'uniprot': 'P35557'}
ENSG00000156515 {'hgnc': 'HGNC:4922', 'hgnc.symbol': 'HK1', 'ensembl': 'ENSG00000156515', 'uniprot': 'P19367'}
ENSG00000160883 {'hgnc': 'HGNC:4925', 'hgnc.symbol': 'HK3', 'ensembl': 'ENSG00000160883', 'uniprot': 'P52790'}
ENSG00000131482 {'hgnc': 'HGNC:4056', 'hgnc.symbol': 'G6PC', 'ensembl': 'ENSG00000131482', 'uniprot': 'P35575'}
ENSG00000105220 {'hgnc': 'HGNC:4458', 'hgnc.symbol': 'GPI', 'ensembl': 'ENSG00000105220', 'uniprot': 'P06744'}
ENSG00000152556 {'hgnc': 'HGNC:8877', 'hgnc.symbol': 'PFKM', 'ensembl': 'ENSG00000152556', 'uniprot': 'P08237'}
ENSG00000165140 {'hgnc': 'HGNC:3606', 'hgnc.symbol': 'FBP1', 'ensembl': 'ENSG00000165140', 'uniprot': 'P09467'}
ENSG00000130957 {'hgnc': 'HGNC:3607', 'hgnc.symbol': 'FBP2', 'ensembl': 'ENSG00000130957', 'uniprot': 'O00757

# Take over the Human1 annotations if not already in Mitocore

In [152]:
for gene_hum in human1.genes:
    human1_id = gene_hum.id

    for gene_mit in mitocore.genes:
        notes = gene_mit.notes

        # check if Human1 mapping exists
        if "Human1" not in notes:
            continue

        if notes["Human1"] != human1_id:
            continue

        # merge annotations: Human1 to MitoCore
        for ann_key, ann_val in gene_hum.annotation.items():
            # only add if MitoCore does NOT already contain the annotation
            if ann_key not in gene_mit.annotation:
                gene_mit.annotation[ann_key] = ann_val

In [153]:
for react in mitocore.genes:
    annotations=react.annotation
    print(react.id, annotations)

ENSG00000159399 {'hgnc': 'HGNC:4923', 'hgnc.symbol': 'HK2', 'ensembl': 'ENSG00000159399', 'uniprot': 'P52789', 'sbo': 'SBO:0000243', 'ncbigene': '3099'}
ENSG00000106633 {'hgnc': 'HGNC:4195', 'hgnc.symbol': 'GCK', 'ensembl': 'ENSG00000106633', 'uniprot': 'P35557', 'sbo': 'SBO:0000243', 'ncbigene': '2645'}
ENSG00000156515 {'hgnc': 'HGNC:4922', 'hgnc.symbol': 'HK1', 'ensembl': 'ENSG00000156515', 'uniprot': 'P19367', 'sbo': 'SBO:0000243', 'ncbigene': '3098'}
ENSG00000160883 {'hgnc': 'HGNC:4925', 'hgnc.symbol': 'HK3', 'ensembl': 'ENSG00000160883', 'uniprot': 'P52790', 'sbo': 'SBO:0000243', 'ncbigene': '3101'}
ENSG00000131482 {'hgnc': 'HGNC:4056', 'hgnc.symbol': 'G6PC', 'ensembl': 'ENSG00000131482', 'uniprot': 'P35575', 'sbo': 'SBO:0000243', 'ncbigene': '2538'}
ENSG00000105220 {'hgnc': 'HGNC:4458', 'hgnc.symbol': 'GPI', 'ensembl': 'ENSG00000105220', 'uniprot': 'P06744', 'sbo': 'SBO:0000243', 'ncbigene': '2821'}
ENSG00000152556 {'hgnc': 'HGNC:8877', 'hgnc.symbol': 'PFKM', 'ensembl': 'ENSG0000

# Indirect accesion of gene species specific KEGG identifiers (API requests)


The KEGG orthology cannot be added directly by API request. Since Mitocore is a Human model, it requires specific Homo-sapiens identifiers (hsa) to send an API request to the KEGG database to retrieve KEGG orthology.

Note: The results of the API request takes long therefore is saves as csv file  (don't repeat it, number of genes shouldn't have changed)


Worflow:
2. Extract uniprot gene ids from model
3. API request to kegg to obtain hsa ids from uniprot ids
4. Save csv file with hsa and uniprot ids 
5. API request to KEGG to obtain KEGG ids from hsa ids

In [154]:


# 2. extract uniprot gene ids from model
uniprot_ids= get_model_ids_for_databases(mitocore, 'genes', '_annotation', ['uniprot'])

# 3. API request to kegg to obtain hsa ids 
import pandas as pd
import time
from urllib import request
# ensure no whitespace or version suffixes
uniprot_ids['uniprot'] = uniprot_ids['uniprot'].astype(str).str.strip()

output_dict = {'uniprot': [], 'hsa': []}
batch_size = 10
uniprot_ids = list(uniprot_ids['uniprot'])

for i in range(0, len(uniprot_ids), batch_size):
    batch = uniprot_ids[i:i + batch_size]
    url = "https://rest.kegg.jp/conv/hsa/" + "+".join([f"uniprot:{uid}" for uid in batch]) #/conv/genes/

    try:
        with request.urlopen(url) as f:
            response = f.read()

        lines = response.decode('utf-8').splitlines()
        print(f"Requested {len(batch)} IDs; got {len(lines)} mappings")

        if lines:
            for line in lines:
                if '\t' in line:
                    uniprot_entry, hsa_id = line.strip().split('\t')
                    uniprot = uniprot_entry.replace("up:", "").strip()
                    hsa = hsa_id.replace("hsa:", "").strip()

                    output_dict['uniprot'].append(uniprot)
                    output_dict['hsa'].append(hsa)

    except Exception as e:
        print(f"Failed on batch starting with {batch[0]}: {e}")
        for uid in batch:
            output_dict['uniprot'].append(uid)
            output_dict['hsa'].append(None)

    time.sleep(3)

# 4.create dataframe with uniprot and hsa ids and save as csv file
hsa_ids_Genes = pd.DataFrame.from_dict(output_dict)
print(f"Final DataFrame has {len(hsa_ids_Genes)} entries")
print(hsa_ids_Genes)



Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 9 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 11 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 11 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 10 mappings
Requested 10 IDs; got 10 mappings
Requested 10 ID

In [155]:
# 5. API request to kegg to obtain kegg gene ids for hsa ids
import pandas as pd
from urllib import request

# Download the full KEGG gene-to-KO mapping for human
url = "http://rest.kegg.jp/link/ko/hsa"
response = request.urlopen(url)
response_content = response.read().decode('utf-8')  # Decode the response content

# Parse the mapping
mapping = []
for line in response_content.strip().split("\n"):
    gene_id, ko_id = line.split("\t")
    mapping.append((gene_id, ko_id.split(":")[1]))  # remove 'ko:' prefix

# Convert to DataFrame
mapping_df = pd.DataFrame(mapping, columns=["hsa", "kegg.genes"])
print(mapping_df)

# Match MIRIAM pattern for kegg.genes
for index, row in mapping_df.iterrows():
    kegg_id = row["kegg.genes"]
    if not kegg_id.startswith("hsa:"):  # Check if prefix is missing
        kegg_id = f"hsa:{kegg_id}"  # Add prefix
        mapping_df.at[index, 'kegg.genes'] = kegg_id
    hsa_id = row["hsa"] 
    if kegg_id.startswith("hsa:"):
        hsa_id = row["hsa"].split(":")[1]
        mapping_df.at[index, 'hsa'] = hsa_id


                 hsa kegg.genes
0             hsa:10     K00622
1            hsa:100     K01488
2           hsa:1000     K06736
3          hsa:10000     K04456
4      hsa:100008587     K01986
...              ...        ...
19038       hsa:9991     K17844
19039       hsa:9992     K04896
19040       hsa:9993     K27941
19041       hsa:9994     K25593
19042       hsa:9997     K23755

[19043 rows x 2 columns]


# Add hsa and KEGG identifiers to the model 

After API request to obtain the hsa orthology and subsequently KEGG identifiers 

Worflow:
2. Extract Uniprot gene ids from model
3. Map uniprot ids to hsa ids
4. Add hsa ids

repeat workflow for KEGG replacing 'Uniprot' through hsa and and run Memote test.

In [156]:
# hsa orthology for genes
# 2. extract uniprot gene ids from model
print('geting ids for all uniprot')
df_genes = get_model_ids_for_databases(mitocore, 'genes', '_annotation', ['uniprot'])
print(df_genes)

# 3. map uniprot ids to hsa ids 
print('Mapping  metabolites to hsa')
df_hsa_genes = map_ids_to_db(df_genes, hsa_ids_Genes,'uniprot', 'uniprot', 'hsa', ',')
print('hsa metabolites \n',df_hsa_genes)

# 4. add hsa ids to model
print('adding hsa ids to the model')
hsa_react = add_ids_to_model(mitocore, df_hsa_genes, 'genes','hsa')

geting ids for all uniprot
            model_id uniprot
0    ENSG00000159399  P52789
1    ENSG00000106633  P35557
2    ENSG00000156515  P19367
3    ENSG00000160883  P52790
4    ENSG00000131482  P35575
..               ...     ...
592  ENSG00000132874  Q15849
593  ENSG00000141469  Q13336
594  ENSG00000110911  P49281
595  ENSG00000018280  P49279
596  ENSG00000138449  Q9NP59

[597 rows x 2 columns]
Mapping  metabolites to hsa
Match found: P52789 -> 3099
Match found: P35557 -> 2645
Match found: P19367 -> 3098
Match found: P52790 -> 3101
Match found: P35575 -> 2538
Match found: P06744 -> 2821
Match found: P08237 -> 5213
Match found: P09467 -> 2203
Match found: O00757 -> 8789
Match found: P09972 -> 230
Match found: P04075 -> 226
Match found: P60174 -> 7167
Match found: P04406 -> 2597
Match found: P00558 -> 5230
Match found: P15259 -> 5224
Match found: P18669 -> 5223
Match found: P13929 -> 2027
Match found: P09104 -> 2026
Match found: P06733 -> 2023
Match found: P14618 -> 5315
Match found: P3

In [157]:
# KEGG identifiers for genes

# 2. extract hsa gene orthology from model
print('geting ids for all uniprot')
df_genes = get_model_ids_for_databases(mitocore, 'genes', '_annotation', ['hsa'])
print(df_genes)

# 3. map hsa orthology to kegg ids 
print('Mapping  metabolites to hsa')
df_kegg_genes = map_ids_to_db(df_genes, mapping_df,'hsa', 'hsa', 'kegg.genes', ',')
print('hsa metabolites \n',df_kegg_genes)

# 4. add hsa ids to model
print('adding kegg ids to the model')
kegg_genes = add_ids_to_model(mitocore, df_kegg_genes, 'genes','kegg.genes')



geting ids for all uniprot
            model_id    hsa
0    ENSG00000159399   3099
1    ENSG00000106633   2645
2    ENSG00000156515   3098
3    ENSG00000160883   3101
4    ENSG00000131482   2538
..               ...    ...
592  ENSG00000132874   8170
593  ENSG00000141469   6563
594  ENSG00000110911   4891
595  ENSG00000018280   6556
596  ENSG00000138449  30061

[597 rows x 2 columns]
Mapping  metabolites to hsa
Match found: 3099 -> hsa:K00844
Match found: 2645 -> hsa:K12407
Match found: 3098 -> hsa:K00844
Match found: 3101 -> hsa:K00844
Match found: 2538 -> hsa:K01084
Match found: 2821 -> hsa:K01810
Match found: 5213 -> hsa:K00850
Match found: 2203 -> hsa:K03841
Match found: 8789 -> hsa:K03841
Match found: 230 -> hsa:K01623
Match found: 226 -> hsa:K01623
Match found: 7167 -> hsa:K01803
Match found: 2597 -> hsa:K00134
Match found: 5230 -> hsa:K00927
Match found: 5224 -> hsa:K01834
Match found: 5223 -> hsa:K01834
Match found: 2027 -> hsa:K01689
Match found: 2026 -> hsa:K01689
Match found

# ------------- Metabolites ------------

Process MetaNetX mapping file 


In [158]:
# 1. load MNX database file for metabolites
mapping_kegg= mnx_processing_file_mnx('/Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/Files_Databases/MNX_chem_xref.tsv','kegg.compound','metabolites',( "kegg.compound:", "keggC:"))
mapping_bigg= mnx_processing_file_mnx('/Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/Files_Databases/MNX_chem_xref.tsv','bigg.metabolite','metabolites',("bigg.metabolite:", "biggM:"))

# 2. merge files into one mapping file
metanetx_mapp_metabolites = pd.merge(mapping_bigg, mapping_kegg, on='metanetx.metabolites', how='outer')
print(metanetx_mapp_metabolites)
#rename columns in dataframe to match MIRIAM pattern
metanetx_mapp_metabolites = metanetx_mapp_metabolites.rename(columns={'bigg.metabolite': 'bigg.metabolite', 'metanetx.metabolites': 'metanetx.chemical', 'kegg.compound': 'kegg.compound'})
print(metanetx_mapp_metabolites)


      bigg.metabolite metanetx.metabolites kegg.compound
0                 oh1               MNXM02        C01328
1                 oh1               MNXM02        C01328
2                 oh1               MNXM02      M_C01328
3               M_oh1               MNXM02        C01328
4               M_oh1               MNXM02        C01328
...               ...                  ...           ...
90571           M_h2o                WATER        C00001
90572           M_h2o                WATER      M_C00001
90573             h2o                WATER        C00001
90574             h2o                WATER        C00001
90575             h2o                WATER      M_C00001

[90576 rows x 3 columns]
      bigg.metabolite metanetx.chemical kegg.compound
0                 oh1            MNXM02        C01328
1                 oh1            MNXM02        C01328
2                 oh1            MNXM02      M_C01328
3               M_oh1            MNXM02        C01328
4               M_oh

workflow add Human1 and MetaNetX identifiers:

2. Extract KEGG and BiGG metabolite ids from model into dataframe
3. Map KEGG ids to Human1 
4. Add Human1 ids 

repeat for MetaNetx ids with Metabolites_metanetx mapping file and Metanetx.chemical instead Human1 and run Memote test

In [159]:

# 2. extract kegg and bigg ids from model into dataframe
print('geting ids for all possible databases ')
df_metab = get_model_ids_for_databases(mitocore, 'metabolites', '_annotation', ['kegg', 'bigg'])
print(df_metab)

# 3. map kegg ids to Human1 metabolites mapping file
print('Mapping metabolites to Human1 ids.')
human1_mets= pd.read_csv('/Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/Files_Databases/Human1_metabolites.tsv', sep='\t')
df_Human1_metab = map_ids_to_db(df_metab, human1_mets,'kegg', 'metKEGGID', 'mets','\t')
print(df_Human1_metab)
#rename columns to match naming convention
df_Human1_metab = df_Human1_metab.rename(columns={'model_id': 'model_id', 'mets': 'Human1'})
print(df_Human1_metab)

# add Human1 ids to model
print('adding Human1 ids to the model')
add_ids_to_model_notes(mitocore, df_Human1_metab,  'metabolites','Human1' )

geting ids for all possible databases 
       model_id    kegg      bigg
0      10fthf_c  C00234    10fthf
1      10fthf_m  C00234    10fthf
2       13dpg_c  C00236     13dpg
3    1pipdn2c_c  C04092  1pipdn2c
4      1pyr5c_m  C03912    1pyr5c
..          ...     ...       ...
440   palmACP_c  C05764   palmACP
441       cbp_c    None      None
442     cbasp_c    None      None
443    dhor_S_c    None      None
444      orot_c    None      None

[445 rows x 3 columns]
Mapping metabolites to Human1 ids.
Match found: C00234 -> MAM00266c
Match found: C00234 -> MAM00266c
Match found: C00236 -> MAM00247c
Match found: C04092 -> MAM01663x
Match found: C03912 -> MAM00559c
Match found: C03508 -> MAM02321m
Match found: C05984 -> MAM00648c
Match found: C05984 -> MAM00648c
Match found: C03344 -> MAM00662m
Match found: C03345 -> MAM02999m
Match found: C01033 -> MAM00663m
Match found: C00349 -> MAM00661c
Match found: C03460 -> MAM02468m
Match found: C00109 -> MAM00671c
Match found: C00109 -> MAM00671c

Name,S1SBMLmodel
Memory address,2a17943d0
Number of metabolites,445
Number of reactions,560
Number of genes,597
Number of groups,96
Objective expression,1.0*OF_ATP_MitoCore - 1.0*OF_ATP_MitoCore_reverse_653d9
Compartments,"Cytosol, Mitochondrion, External"


In [160]:

# 2. extract kegg and bigg ids from model into dataframe
print('geting ids for all possible databases ')
df_metab = get_model_ids_for_databases(mitocore, 'metabolites', '_annotation', ['kegg.compound', 'bigg.metabolite'])
print(df_metab)

# 3. map kegg ids to metanetx ids metabolites mapping file
print('Mapping  metabolites to metanetx.')
df_MNX_metab = map_ids_to_db(df_metab, metanetx_mapp_metabolites,'kegg.compound', 'kegg.compound', 'metanetx.chemical', ',')
print('MNX metabolites \n',df_MNX_metab)

# add metanetx ids to model
print('adding MNX ids to the model')
MNX_metab = add_ids_to_model(mitocore, df_MNX_metab,  'metabolites','metanetx.chemical' )


geting ids for all possible databases 
       model_id kegg.compound bigg.metabolite
0      10fthf_c        C00234          10fthf
1      10fthf_m        C00234          10fthf
2       13dpg_c        C00236           13dpg
3    1pipdn2c_c        C04092        1pipdn2c
4      1pyr5c_m        C03912          1pyr5c
..          ...           ...             ...
440   palmACP_c        C05764         palmACP
441       cbp_c          None            None
442     cbasp_c          None            None
443    dhor_S_c          None            None
444      orot_c          None            None

[445 rows x 3 columns]
Mapping  metabolites to metanetx.
Match found: C00234 -> MNXM1102376
Match found: C00234 -> MNXM1102376
Match found: C00236 -> MNXM1108073
Match found: C04092 -> MNXM911
Match found: C03912 -> MNXM1105060
Match found: C03508 -> MNXM114087
Match found: C05984 -> MNXM1103784
Match found: C05984 -> MNXM1103784
Match found: C03344 -> MNXM1104585
Match found: C03345 -> MNXM1104649
Match 

# Take over annotations from Human1 if not alredy included in MitoCore

In [161]:
for met_hum in human1.metabolites:
    human1_id = met_hum.id

    for met_mit in mitocore.metabolites:
        notes = met_mit.notes

        # check if Human1 mapping exists
        if "Human1" not in notes:
            continue

        if notes["Human1"] != human1_id:
            continue

        # merge annotations: Human1 to MitoCore
        for ann_key, ann_val in met_hum.annotation.items():
            # only add if MitoCore does NOT already contain the annotation
            if ann_key not in met_mit.annotation:
                met_mit.annotation[ann_key] = ann_val

In [162]:
#check the notes
for react in mitocore.metabolites:
    notes=react.annotation
    print(react.id, notes)

10fthf_c {'bigg.metabolite': '10fthf', 'kegg.compound': 'C00234', 'metanetx.chemical': 'MNXM1102376', 'sbo': 'SBO:0000247', 'chebi': 'CHEBI:15637', 'pubchem.compound': '122347', 'vmhmetabolite': '10fthf'}
10fthf_m {'bigg.metabolite': '10fthf', 'kegg.compound': 'C00234', 'metanetx.chemical': 'MNXM1102376', 'sbo': 'SBO:0000247', 'chebi': 'CHEBI:15637', 'pubchem.compound': '122347', 'vmhmetabolite': '10fthf'}
13dpg_c {'bigg.metabolite': '13dpg', 'kegg.compound': 'C00236', 'metanetx.chemical': 'MNXM1108073', 'sbo': 'SBO:0000247', 'chebi': 'CHEBI:16001', 'pubchem.compound': '439191', 'vmhmetabolite': '13dpg'}
1pipdn2c_c {'bigg.metabolite': '1pipdn2c', 'kegg.compound': 'C04092', 'metanetx.chemical': 'MNXM911', 'sbo': 'SBO:0000247', 'hmdb': 'HMDB0001084', 'chebi': 'CHEBI:30912', 'pubchem.compound': '1194', 'vmhmetabolite': '1pipdn2c'}
1pyr5c_m {'bigg.metabolite': '1pyr5c', 'kegg.compound': 'C03912', 'metanetx.chemical': 'MNXM1105060', 'sbo': 'SBO:0000247', 'chebi': 'CHEBI:371', 'pubchem.compo

In [163]:
# Clean N/A keys added during annotation curation in previous steps
mitocore = clean_invalid_annotations(mitocore)

# save cleaned model as new SBML file
cobra.io.write_sbml_model(mitocore, "/Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/MitoCore_Curation/Mitocore_aligned_to_Human1.xml")

# Run memote snapshot on the corrected model
!memote report snapshot --filename "/Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/MitoCore_Curation/Mitocore_aligned_to_Human1.html" /Users/benjaminreyes/Desktop/Masterarbeit/neutrophil_modeling/neutrophil-modeling/curation/MitoCore_Curation/Mitocore_aligned_to_Human1.xml
# ---End---

The current solver interface glpk doesn't support setting the optimality tolerance.
============================= test session starts ==============================
platform darwin -- Python 3.10.8, pytest-7.1.2, pluggy-1.0.0
rootdir: /Users/benjaminreyes
plugins: anyio-3.5.0, typeguard-4.4.2
collected 155 items / 1 skipped                                                

../../../../../../anaconda3/lib/python3.10/site-packages/memote/suite/tests/test_annotation.py F [  0%]
F.FFFFFFFFFFFFFFFFFFFFFFFFFFFFFF..FF..FFF.F.......F.F.F.F.FFFF..         [ 41%]
../../../../../../anaconda3/lib/python3.10/site-packages/memote/suite/tests/test_basic.py . [ 42%]
.....FFF......F.F.F.FF                                                   [ 56%]
../../../../../../anaconda3/lib/python3.10/site-packages/memote/suite/tests/test_biomass.py . [ 57%]
FFFFFF....FFFFF.FF                                                       [ 69%]
../../../../../../anaconda3/lib/python3.10/site-packages/memote/suite/tests/test_